In [ ]:
pip install opencv-python mediapipe pandas numpy

In [ ]:
import os

data_dir = 'path/to/Dataset Isyarat'

for folder in os.listdir(data_dir):
    folder_path = os.path.join(data_dir, folder)
    if os.path.isdir(folder_path):
        print(f'Kelas: {folder}')
        for file in os.listdir(folder_path):
            print(f'  - {file}')

In [ ]:
pip install opencv-python

In [ ]:
import cv2

In [ ]:
import cv2
import os

def extract_frames(video_path, output_dir, target_frame_count=30):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    interval = max(total_frames // target_frame_count, 1)

    video_name = os.path.splitext(os.path.basename(video_path))[0]
    count = 0
    frame_idx = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if count % interval == 0 and frame_idx < target_frame_count:
            filename = f"{video_name}_frame{frame_idx:03d}.jpg"
            output_path = os.path.join(output_dir, filename)
            cv2.imwrite(output_path, frame)
            frame_idx += 1
        count += 1
    cap.release()

In [ ]:
# 1️⃣ Tentukan folder dataset video
data_dir = r'path/to/Dataset Isyarat'  # Ganti sesuai lokasi asli

# 2️⃣ Tentukan folder output frame
output_root = 'frames'  # folder hasil frame, akan dibuat otomatis

# 3️⃣ Loop untuk proses semua video di setiap folder kelas
for class_folder in os.listdir(data_dir):
    class_path = os.path.join(data_dir, class_folder)
    if os.path.isdir(class_path):
        output_dir = os.path.join(output_root, class_folder)
        os.makedirs(output_dir, exist_ok=True)
        for video_file in os.listdir(class_path):
            if video_file.endswith('.mp4'):
                video_path = os.path.join(class_path, video_file)
                extract_frames(video_path, output_dir, target_frame_count=30)
                print(f'✅ Frame diekstrak dari: {video_path}')


In [ ]:
pip install mediapipe

In [ ]:
import cv2
import mediapipe as mp
import os
import pandas as pd

# Inisialisasi modul MediaPipe
mp_pose = mp.solutions.pose
mp_hands = mp.solutions.hands
mp_face = mp.solutions.face_mesh
pose = mp_pose.Pose(static_image_mode=True)
hands = mp_hands.Hands(static_image_mode=True, max_num_hands=2)
face = mp_face.FaceMesh(static_image_mode=True)

POSE_CONNECTIONS = mp.solutions.pose.POSE_CONNECTIONS
HAND_CONNECTIONS = mp.solutions.hands.HAND_CONNECTIONS
FACE_CONNECTIONS = [(i, i + 1) for i in range(19)]  # Sederhana, hanya wajah 0–19

# Fungsi ekstraksi fitur dari 1 gambar
def extract_landmarks(image_path):
    image = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    results_pose = pose.process(image_rgb)
    results_hands = hands.process(image_rgb)
    results_face = face.process(image_rgb)

    features = []

    # 1. Pose (33 keypoints)
    if results_pose.pose_landmarks:
        for lm in results_pose.pose_landmarks.landmark:
            features.extend([lm.x, lm.y, lm.z])
    else:
        features.extend([0]*99)  # 33 * 3

    # 2. Hands (21 keypoints per tangan × 2)
    if results_hands.multi_hand_landmarks:
        for hand in results_hands.multi_hand_landmarks:
            for lm in hand.landmark:
                features.extend([lm.x, lm.y, lm.z])
        if len(results_hands.multi_hand_landmarks) == 1:
            features.extend([0]*63)  # tangan satunya kosong
    else:
        features.extend([0]*126)

    # 3. Face (ambil 20 landmark awal untuk ringkasan ekspresi/bibir)
    if results_face.multi_face_landmarks:
        for lm in results_face.multi_face_landmarks[0].landmark[:20]:
            features.extend([lm.x, lm.y, lm.z])
    else:
        features.extend([0]*60)

    return features


In [ ]:
# Lokasi folder berisi frame hasil ekstrak dari video
frames_dir = 'frames' 
output_csv = 'dataset_keypoints.csv'  # Output dataset

data = []

# Loop semua folder kelas dan gambar .jpg
for class_folder in os.listdir(frames_dir):
    class_path = os.path.join(frames_dir, class_folder)
    if os.path.isdir(class_path):
        print(f'🔍 Memproses kelas: {class_folder}')
        for img_file in os.listdir(class_path):
            if img_file.endswith('.jpg'):
                img_path = os.path.join(class_path, img_file)
                features = extract_landmarks(img_path)
                features.append(class_folder)  # tambahkan label kelas
                data.append(features)
                print(f'✅ Ekstraksi selesai: {img_path}')

# Simpan ke CSV
columns = [f'f{i}' for i in range(len(data[0])-1)] + ['label']
df = pd.DataFrame(data, columns=columns)
df.to_csv(output_csv, index=False)
print(f'\n📁 Dataset fitur berhasil disimpan sebagai: {output_csv}')


In [ ]:
!pip install pandas matplotlib


In [ ]:
import cv2
import numpy as np
import os

# === Koneksi Skeleton ===
POSE_CONNECTIONS = [
    (11, 13), (13, 15), (15, 17), (17, 19), (19, 21),
    (12, 14), (14, 16), (16, 18), (18, 20), (20, 22),
    (11, 12), (11, 23), (12, 24),
    (23, 24), (23, 25), (24, 26),
    (25, 27), (27, 29), (26, 28), (28, 30),
]

LEFT_HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (0, 9), (9,10), (10,11), (11,12),
    (0,13), (13,14), (14,15), (15,16),
    (0,17), (17,18), (18,19), (19,20),
]

FACE_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (5, 6), (6, 7), (7, 8), (8, 9),
    (10, 11), (11, 12), (12, 13),
    (14, 15), (15, 16), (16, 17), (17, 18),
    (19, 20), (20, 21), (21, 22), (22, 23)
]

FACE_OFFSET = 75  # Indeks awal keypoint wajah (sesuai urutan mediapipe holistic)

# === Fungsi Visualisasi ===
def visualize_keypoints_with_lines(row, save_path=None):
    label = str(row.iloc[-1])
    data_only = row.iloc[:-1]

    try:
        coords = np.array(data_only, dtype=np.float32).reshape(-1, 3)
    except:
        print(f"❌ Gagal reshape: {len(data_only)} kolom → tidak valid.")
        return

    img = np.ones((480, 640, 3), dtype=np.uint8) * 255  # canvas putih

    # Gambar titik koordinat
    for i, point in enumerate(coords):
        x, y, z = point
        if 0 < x < 1 and 0 < y < 1:
            x_px = int(x * 640)
            y_px = int(y * 480)
            cv2.circle(img, (x_px, y_px), 3, (0, 0, 255), -1)

    # Gambar garis POSE
    for start, end in POSE_CONNECTIONS:
        if start < len(coords) and end < len(coords):
            x1, y1, _ = coords[start]
            x2, y2, _ = coords[end]
            if all(0 < v < 1 for v in [x1, y1, x2, y2]):
                cv2.line(img, (int(x1 * 640), int(y1 * 480)),
                         (int(x2 * 640), int(y2 * 480)), (0, 255, 0), 2)

    # Gambar garis tangan kiri & kanan
    offset_left = 33
    offset_right = 54

    for start, end in LEFT_HAND_CONNECTIONS:
        # Kiri
        s_idx, e_idx = offset_left + start, offset_left + end
        if s_idx < len(coords) and e_idx < len(coords):
            x1, y1, _ = coords[s_idx]
            x2, y2, _ = coords[e_idx]
            if all(0 < v < 1 for v in [x1, y1, x2, y2]):
                cv2.line(img, (int(x1 * 640), int(y1 * 480)),
                         (int(x2 * 640), int(y2 * 480)), (255, 0, 0), 1)

        # Kanan
        s_idx, e_idx = offset_right + start, offset_right + end
        if s_idx < len(coords) and e_idx < len(coords):
            x1, y1, _ = coords[s_idx]
            x2, y2, _ = coords[e_idx]
            if all(0 < v < 1 for v in [x1, y1, x2, y2]):
                cv2.line(img, (int(x1 * 640), int(y1 * 480)),
                         (int(x2 * 640), int(y2 * 480)), (255, 100, 0), 1)

    # Gambar garis wajah
    for start, end in FACE_CONNECTIONS:
        s_idx, e_idx = FACE_OFFSET + start, FACE_OFFSET + end
        if s_idx < len(coords) and e_idx < len(coords):
            x1, y1, _ = coords[s_idx]
            x2, y2, _ = coords[e_idx]
            if all(0 < v < 1 for v in [x1, y1, x2, y2]):
                cv2.line(img, (int(x1 * 640), int(y1 * 480)),
                         (int(x2 * 640), int(y2 * 480)), (150, 0, 150), 1)

    # Tambah label kelas
    cv2.putText(img, f"Class: {label}", (10, 25),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 2)

    # Simpan
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        cv2.imwrite(save_path, img)


In [ ]:
import pandas as pd

df = pd.read_csv("dataset_keypoints.csv")
for idx, row in df.iterrows():
    save_path = f"visualized_frames/frame_{idx:04d}.jpg"
    visualize_keypoints_with_lines(row, save_path)


In [ ]:
import cv2
import os

# Path folder gambar
frames_folder = 'visualized_frames'
output_video = 'skeleton_video.mp4'

# Ambil semua nama file dan urutkan
frame_files = sorted([
    f for f in os.listdir(frames_folder) if f.endswith('.jpg')
])

# Ambil dimensi dari salah satu frame
first_frame = cv2.imread(os.path.join(frames_folder, frame_files[0]))
height, width, _ = first_frame.shape

# Buat writer video
out = cv2.VideoWriter(output_video, cv2.VideoWriter_fourcc(*'mp4v'), 10, (width, height))

for file in frame_files:
    frame = cv2.imread(os.path.join(frames_folder, file))
    out.write(frame)

out.release()
print(f"✅ Video disimpan sebagai: {output_video}")


In [ ]:
pip install imageio


In [ ]:
import imageio
import os

frames_folder = 'visualized_frames'
output_gif = 'skeleton_animation.gif'

frame_files = sorted([
    f for f in os.listdir(frames_folder) if f.endswith('.jpg')
])

images = []
for file in frame_files:
    img_path = os.path.join(frames_folder, file)
    images.append(imageio.imread(img_path))

# Simpan sebagai GIF, durasi per frame = 0.1 detik
imageio.mimsave(output_gif, images, duration=0.1)
print(f"✅ GIF disimpan sebagai: {output_gif}")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load dataset
df = pd.read_csv('dataset_keypoints.csv')

def plot_keypoints_with_connections(row, ax, title=''):
    data = row.values.astype(float)
    coords = data.reshape(-1, 3)
    xs = coords[:, 0]
    ys = coords[:, 1]

    ax.scatter(xs, ys, c='blue', s=10)

    # POSE
    pose_connections = [
        (11, 13), (13, 15), (15, 17), (17, 19), (19, 21),  # left arm
        (12, 14), (14, 16), (16, 18), (18, 20), (20, 22),  # right arm
        (11, 12), (11, 23), (12, 24), (23, 24),            # torso
        (23, 25), (25, 27), (27, 29),                      # left leg
        (24, 26), (26, 28), (28, 30),                      # right leg
    ]
    for s, e in pose_connections:
        ax.plot([xs[s], xs[e]], [ys[s], ys[e]], color='green', linewidth=1)

    # HAND CONNECTIONS
    hand_connections = [
        (0, 1), (1, 2), (2, 3), (3, 4),      # Thumb
        (0, 5), (5, 6), (6, 7), (7, 8),      # Index
        (0, 9), (9,10), (10,11), (11,12),    # Middle
        (0,13), (13,14), (14,15), (15,16),   # Ring
        (0,17), (17,18), (18,19), (19,20)    # Pinky
    ]
    # kiri
    offset_l = 33
    # kanan
    offset_r = 54
    for s, e in hand_connections:
        ax.plot([xs[offset_l + s], xs[offset_l + e]],
                [ys[offset_l + s], ys[offset_l + e]],
                color='orange', linewidth=1)
        ax.plot([xs[offset_r + s], xs[offset_r + e]],
                [ys[offset_r + s], ys[offset_r + e]],
                color='purple', linewidth=1)

    # FACE CONNECTIONS
    face_offset = 75
    for i in range(19):
        ax.plot([xs[face_offset + i], xs[face_offset + i + 1]],
                [ys[face_offset + i], ys[face_offset + i + 1]],
                color='red', linewidth=1)

    ax.set_title(title)
    ax.invert_yaxis()
    ax.axis('off')

# Visualisasi 1 sampel dari setiap kelas
labels = df['label'].unique()
fig, axes = plt.subplots(1, len(labels), figsize=(5 * len(labels), 5))

if len(labels) == 1:
    axes = [axes]

for i, label in enumerate(labels):
    row = df[df['label'] == label].iloc[0]
    row_data = row[:-1]  # tanpa label
    plot_keypoints_with_connections(row_data, axes[i], title=f"Label: {label}")

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd

# Load dataset dari file CSV
df = pd.read_csv('dataset_keypoints.csv')

# Tampilkan 10 baris pertama
print(df.head(5))


In [ ]:
pip install pandas matplotlib pillow


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np
import os

# Pastikan folder output GIF tersedia
os.makedirs("gif_output", exist_ok=True)

# Load dataset
df = pd.read_csv('dataset_keypoints.csv')
labels = df['label'].unique()

# Skeleton connection setup
POSE_CONNECTIONS = [
    (11, 13), (13, 15), (15, 17), (17, 19), (19, 21),
    (12, 14), (14, 16), (16, 18), (18, 20), (20, 22),
    (11, 12), (11, 23), (12, 24), (23, 24),
    (23, 25), (25, 27), (27, 29), (24, 26), (26, 28), (28, 30)
]

HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (0, 9), (9,10), (10,11), (11,12),
    (0,13), (13,14), (14,15), (15,16),
    (0,17), (17,18), (18,19), (19,20)
]

def plot_frame(data, ax, label=''):
    coords = np.array(data, dtype=float).reshape(-1, 3)
    xs = coords[:, 0]
    ys = coords[:, 1]

    ax.clear()
    ax.scatter(xs, ys, c='blue', s=10)

    for s, e in POSE_CONNECTIONS:
        ax.plot([xs[s], xs[e]], [ys[s], ys[e]], color='green', linewidth=1)

    offset_l = 33
    offset_r = 54
    for s, e in HAND_CONNECTIONS:
        if offset_l + e < len(xs) and offset_r + e < len(xs):
            ax.plot([xs[offset_l + s], xs[offset_l + e]],
                    [ys[offset_l + s], ys[offset_l + e]],
                    color='orange', linewidth=1)
            ax.plot([xs[offset_r + s], xs[offset_r + e]],
                    [ys[offset_r + s], ys[offset_r + e]],
                    color='purple', linewidth=1)

    face_offset = 75
    for i in range(19):
        if face_offset + i + 1 < len(xs):
            ax.plot([xs[face_offset + i], xs[face_offset + i + 1]],
                    [ys[face_offset + i], ys[face_offset + i + 1]],
                    color='red', linewidth=1)

    ax.set_title(f"Label: {label}")
    ax.invert_yaxis()
    ax.axis('off')

def save_class_gif(class_label):
    df_class = df[df['label'] == class_label].reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(5, 5))

    def update(frame_idx):
        row = df_class.iloc[frame_idx][:-1]
        plot_frame(row, ax, label=class_label)

    ani = animation.FuncAnimation(fig, update, frames=len(df_class), interval=200)

    gif_path = f"gif_output/{class_label}.gif"
    ani.save(gif_path, writer='pillow')
    plt.close()
    print(f"✅ GIF disimpan: {gif_path}")

# Jalankan untuk semua kelas
for label in labels:
    save_class_gif(label)


In [ ]:
pip install seaborn


In [ ]:
import seaborn as sns


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Load dataset keypoints
df = pd.read_csv('dataset_keypoints.csv')

# Hitung dimensi
n_dim = df.shape[1] - 1  # tanpa label
n_points = n_dim // 3    # x, y, z untuk tiap keypoint

# Tentukan indeks segmen
pose_idx = range(0, 33 * 3)               # 0–98
lh_idx = range(33 * 3, (33 + 21) * 3)     # 99–161
rh_idx = range((33 + 21) * 3, (33 + 42) * 3)  # 162–224
face_idx = range((33 + 42) * 3, n_dim)    # 225–284

# Fungsi untuk menghitung missing pada segmen
def count_missing(segment_idx, row):
    values = row.iloc[list(segment_idx)].values.astype(float)
    reshaped = values.reshape(-1, 3)
    missing = np.sum((reshaped[:, 0] == 0) & (reshaped[:, 1] == 0))
    return missing

# Buat dataframe hasil validasi
missing_summary = df.drop(columns=['label']).apply(lambda row: pd.Series({
    'missing_pose': count_missing(pose_idx, row),
    'missing_left_hand': count_missing(lh_idx, row),
    'missing_right_hand': count_missing(rh_idx, row),
    'missing_face': count_missing(face_idx, row),
}), axis=1)

# Gabungkan dengan label
missing_summary['label'] = df['label']

# --- 🎨 Visualisasi ---
# --- 📊 Heatmap gabungan (overview) ---
plt.figure(figsize=(12, 6))
sns.heatmap(missing_summary.drop(columns=['label']),
            cmap='YlOrRd',
            vmin=0, vmax=33,  # set agar kontras untuk pose/tangan
            cbar_kws={'label': 'Missing Keypoints'},
            linewidths=0.5)
plt.title('Heatmap of Missing Keypoints per Segment (Overview)')
plt.xlabel('Skeleton Segment')
plt.ylabel('Frame Index')
plt.tight_layout()
plt.show()

# --- 🎯 Heatmap detail per segmen ---
segment_info = {
    'missing_pose': 33,
    'missing_left_hand': 21,
    'missing_right_hand': 21,
    'missing_face': 468
}

fig, axes = plt.subplots(nrows=4, ncols=1, figsize=(12, 8), sharex=True)

for ax, (seg, vmax_val) in zip(axes, segment_info.items()):
    sns.heatmap(missing_summary[[seg]].T,
                cmap='YlOrRd',
                vmin=0, vmax=vmax_val,
                cbar_kws={'label': 'Missing Keypoints'},
                ax=ax)
    ax.set_ylabel(seg.replace('_', ' ').title(), rotation=0, labelpad=50)

axes[-1].set_xlabel('Frame Index')
plt.suptitle('Heatmap Missing Keypoints per Segment (Detail)', y=1.02)
plt.tight_layout()
plt.show()
# Heatmap missing per frame (baris = frame, kolom = segmen)


# --- 📊 Histogram distribusi missing pose/hand/face ---
plt.figure(figsize=(14, 4))
for i, col in enumerate(['missing_pose', 'missing_left_hand', 'missing_right_hand', 'missing_face']):
    plt.subplot(1, 4, i+1)
    sns.histplot(missing_summary[col], bins=15, kde=False)
    plt.title(col.replace('_', ' ').title())
    plt.xlabel("Missing")
    plt.ylabel("Frames")
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv('dataset_keypoints.csv')

# Jumlah total keypoints per segmen (dikali 3 karena x,y,z)
POSE_KP = 33 * 3
LH_KP   = 21 * 3
RH_KP   = 21 * 3
FACE_KP = 20 * 3

# Offset posisi
POSE_IDX = slice(0, POSE_KP)
LH_IDX   = slice(POSE_KP, POSE_KP + LH_KP)
RH_IDX   = slice(POSE_KP + LH_KP, POSE_KP + LH_KP + RH_KP)
FACE_IDX = slice(POSE_KP + LH_KP + RH_KP, POSE_KP + LH_KP + RH_KP + FACE_KP)

# Fungsi menghitung jumlah missing (semua 0 di x,y,z)
def count_missing(segment):
    reshaped = segment.reshape(-1, 3)
    return np.sum(np.all(reshaped == 0, axis=1))

# Hitung missing per frame
missing_counts = []
for i, row in df.iterrows():
    values = row[:-1].values.astype(float)  # exclude label
    pose = values[POSE_IDX]
    lh = values[LH_IDX]
    rh = values[RH_IDX]
    face = values[FACE_IDX]
    total_missing = (
        count_missing(pose) +
        count_missing(lh) +
        count_missing(rh) +
        count_missing(face)
    )
    missing_counts.append(total_missing)

df['missing_total'] = missing_counts

# Tampilkan distribusi missing
print(df['missing_total'].describe())

# Buang frame dengan missing lebih dari threshold
threshold = 30  # misalnya: 30 keypoint (dari total 107)
clean_df = df[df['missing_total'] <= threshold].drop(columns=['missing_total'])

print(f"✅ Dataset bersih: {clean_df.shape[0]} frame dari {df.shape[0]}")

# Simpan ke file baru
clean_df.to_csv('dataset_keypoints_clean.csv', index=False)


In [ ]:
# Load data bersih
df_clean = pd.read_csv('dataset_keypoints_clean.csv')

# Misal: kita imputasi hanya nilai 0 (missing) → diganti dengan rata-rata dari kelasnya
features = df_clean.columns[:-1]  # semua kecuali label
labels = df_clean['label'].unique()

for label in labels:
    subset = df_clean[df_clean['label'] == label]
    mean_values = subset[features].replace(0, np.nan).mean(skipna=True)

    # Imputasi nilai 0 → rata-rata per kolom di kelas tersebut
    mask = (df_clean['label'] == label)
    df_clean.loc[mask, features] = df_clean.loc[mask, features].replace(0, np.nan)
    df_clean.loc[mask, features] = df_clean.loc[mask, features].fillna(mean_values)

print("✅ Imputasi selesai. Dataset siap digunakan.")

# Simpan hasil akhir ke file
df_clean.to_csv('dataset_keypoints_final.csv', index=False)


In [ ]:
import pandas as pd

df = pd.read_csv('dataset_keypoints_final.csv')


In [ ]:
import pandas as pd

# Load dataset dari file CSV
df = pd.read_csv('dataset_keypoints_final.csv')

# Tampilkan 5 baris pertama
print(df.head(5))

In [ ]:
X = df.iloc[:, :-1].values  # Semua kolom kecuali kolom terakhir
y = df.iloc[:, -1].values   # Kolom terakhir adalah label isyarat


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# 1. Load final dataset
df = pd.read_csv('dataset_keypoints_final.csv')

# 2. Pisahkan fitur dan label
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

# 3. Encode label ke angka
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# 4. Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"Train: {X_train.shape} → {y_train.shape}")
print(f"Test : {X_test.shape} → {y_test.shape}")


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import pandas as pd

# Load dataset
df = pd.read_csv("dataset_keypoints_final.csv")

# Pisahkan fitur dan label
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

# Label encoding
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Split 1: 80% Train, 20% Test
X_train_80, X_test_20, y_train_80, y_test_20 = train_test_split(
    X, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42
)

# Split 2: 80% Train → 10% Val + 10% Test
X_temp_80, X_test_10, y_temp_80, y_test_10 = train_test_split(
    X_train_80, y_train_80, test_size=0.125, stratify=y_train_80, random_state=42
)  # 0.125 dari 80% = 10%
X_train_80_2, X_val_10 = X_temp_80, X_test_10
y_train_80_2, y_val_10 = y_temp_80, y_test_10

# Split 3: 70% Train, 30% Test
X_train_70, X_test_30, y_train_70, y_test_30 = train_test_split(
    X, y_encoded, test_size=0.3, stratify=y_encoded, random_state=42
)

# Split 4: 70% Train → 20% Val + 10% Test
X_temp_70, X_test_10b, y_temp_70, y_test_10b = train_test_split(
    X_train_70, y_train_70, test_size=2/7, stratify=y_train_70, random_state=42
)  # 2/7 dari 70% = 20%
X_train_70_2, X_val_20 = X_temp_70, X_test_10b
y_train_70_2, y_val_20 = y_temp_70, y_test_10b


In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Buat model MLP
mlp = MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=500, random_state=42)
mlp.fit(X_train, y_train)

# Prediksi dan evaluasi
y_pred = mlp.predict(X_test)

print("📊 Classification Report (MLP):")
print(classification_report(y_test, y_pred, target_names=le.classes_))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title("Confusion Matrix - MLP")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


In [ ]:
from sklearn.svm import SVC

svm = SVC(kernel='rbf', C=1.0, gamma='scale')
svm.fit(X_train, y_train)

y_pred_svm = svm.predict(X_test)
print("📊 Classification Report (SVM):")
print(classification_report(y_test, y_pred_svm, target_names=le.classes_))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title("Confusion Matrix - SVM")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
# Load data
df = pd.read_csv('dataset_keypoints_final.csv')
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

# Encode label
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42
)


In [ ]:
models = {
    "MLP": MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=500, random_state=42),
    "SVM": SVC(kernel='rbf', probability=True, random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5)
}


In [ ]:
results = {}

for name, model in models.items():
    print(f"\n📌 Training: {name}")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    print(f"✅ Accuracy {name}: {acc:.4f}")
    print(classification_report(y_test, y_pred, target_names=le.classes_))
    
    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=le.classes_, yticklabels=le.classes_)
    plt.title(f'Confusion Matrix - {name}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.show()
    
    results[name] = {
        "model": model,
        "accuracy": acc
    }


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# Misal cm = confusion matrix hasil prediksi
# cm = confusion_matrix(y_true, y_pred)

# Label baru
labels = ["How are you", "Hello", "Sorry", "You're welcome", "Thank you", "Help"]

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels,
            yticklabels=labels)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - KNN")
plt.show()


In [ ]:
import pandas as pd

for fname in [
    "dataset_keypoints.csv",
    "dataset_keypoints_clean.csv",
    "dataset_keypoints_final.csv"
]:
    try:
        df = pd.read_csv(fname)
        print(f"\n=== {fname} ===")
        print(df.columns)
    except FileNotFoundError:
        print(f"{fname} tidak ada")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# === Masukkan confusion matrix hasil dari sklearn.metrics.confusion_matrix ===
# Contoh angka di bawah hanya dummy, ganti dengan data asli kamu
cm_knn = [
    [179, 0, 5, 2, 0, 2],
    [1, 167, 5, 3, 1, 1],
    [0, 5, 154, 7, 0, 12],
    [0, 1, 11, 164, 2, 2],
    [0, 6, 5, 3, 161, 0],
    [0, 3, 4, 4, 0, 166]
]

cm_mlp = [
    [185, 0, 1, 0, 1, 1],
    [0, 171, 0, 0, 7, 0],
    [0, 0, 162, 15, 0, 1],
    [0, 0, 0, 179, 1, 0],
    [0, 0, 0, 0, 175, 0],
    [0, 0, 1, 6, 0, 170]
]

cm_rf = [
    [188, 0, 0, 0, 0, 0],
    [0, 178, 0, 0, 0, 0],
    [0, 0, 174, 2, 0, 2],
    [0, 0, 0, 180, 0, 0],
    [0, 0, 0, 0, 175, 0],
    [0, 0, 0, 0, 0, 177]
]

cm_svm = [
    [175, 0, 5, 5, 1, 2],
    [0, 147, 0, 22, 8, 1],
    [0, 5, 144, 28, 0, 1],
    [0, 0, 0, 180, 0, 0],
    [0, 0, 0, 12, 163, 0],
    [0, 0, 0, 27, 0, 150]
]

# === Label kelas dalam bahasa Inggris ===
labels_en = ["How are you", "Hello", "Sorry", "You're welcome", "Thank you", "Help"]

# === Fungsi untuk menggambar confusion matrix ===
def plot_cm(cm, title):
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels_en,
                yticklabels=labels_en)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(title)
    plt.show()

# === Tampilkan semua confusion matrix ===
plot_cm(cm_knn, "Confusion Matrix - KNN")
plot_cm(cm_mlp, "Confusion Matrix - MLP")
plot_cm(cm_rf, "Confusion Matrix - RandomForest")
plot_cm(cm_svm, "Confusion Matrix - SVM")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# === Masukkan confusion matrix hasil dari sklearn.metrics.confusion_matrix ===
# Ganti ini dengan hasil asli kamu
cm_knn = [
    [179, 0, 5, 2, 0, 2],
    [1, 167, 5, 3, 1, 1],
    [0, 5, 154, 7, 0, 12],
    [0, 1, 11, 164, 2, 2],
    [0, 6, 5, 3, 161, 0],
    [0, 3, 4, 4, 0, 166]
]

cm_mlp = [
    [185, 0, 1, 0, 1, 1],
    [0, 171, 0, 0, 7, 0],
    [0, 0, 162, 15, 0, 1],
    [0, 0, 0, 179, 1, 0],
    [0, 0, 0, 0, 175, 0],
    [0, 0, 1, 6, 0, 170]
]

cm_rf = [
    [188, 0, 0, 0, 0, 0],
    [0, 178, 0, 0, 0, 0],
    [0, 0, 174, 2, 0, 2],
    [0, 0, 0, 180, 0, 0],
    [0, 0, 0, 0, 175, 0],
    [0, 0, 0, 0, 0, 177]
]

cm_svm = [
    [175, 0, 5, 5, 1, 2],
    [0, 147, 0, 22, 8, 1],
    [0, 5, 144, 28, 0, 1],
    [0, 0, 0, 180, 0, 0],
    [0, 0, 0, 12, 163, 0],
    [0, 0, 0, 27, 0, 150]
]

# === Label kelas dalam bahasa Inggris ===
labels_en = ["How are you", "Hello", "Sorry", "You're welcome", "Thank you", "Help"]

# === Setup plot grid 2x2 dengan jarak lebih lebar ===
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.subplots_adjust(hspace=0.6, wspace=0.5)  # jarak vertikal & horizontal diperbesar

# List matriks dan judulnya
cms = [(cm_knn, "KNN"), (cm_mlp, "MLP"), (cm_rf, "RandomForest"), (cm_svm, "SVM")]

# Loop untuk plot
for ax, (cm, title) in zip(axes.flat, cms):
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels_en,
                yticklabels=labels_en,
                ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(f"Confusion Matrix - {title}")

plt.show() 


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

# === 1. Load dataset ===
df = pd.read_csv('dataset_keypoints_final.csv')
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

# Encode label kelas BISINDO
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Train-test split (80:20)
X_train, X_val, y_train, y_val = train_test_split(
    X, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42
)

# === 2. Training MLP dengan warm_start agar bisa rekam per epoch ===
mlp = MLPClassifier(
    hidden_layer_sizes=(256, 128),
    max_iter=1,   # latih per 1 iterasi
    warm_start=True,
    random_state=42
)

epochs = 50
train_loss, train_acc, val_acc = [], [], []

for epoch in range(epochs):
    mlp.fit(X_train, y_train)

    # Rekam training loss (sklearn otomatis simpan di loss_)
    train_loss.append(mlp.loss_)

    # Hitung akurasi training & validation
    y_train_pred = mlp.predict(X_train)
    y_val_pred = mlp.predict(X_val)
    train_acc.append(accuracy_score(y_train, y_train_pred))
    val_acc.append(accuracy_score(y_val, y_val_pred))

    print(f"Epoch {epoch+1}/{epochs} - Loss: {mlp.loss_:.4f} | "
          f"Train Acc: {train_acc[-1]:.4f} | Val Acc: {val_acc[-1]:.4f}")

# === 3. Plot Training & Validation Accuracy ===
plt.figure(figsize=(12,5))

# Plot Accuracy
plt.subplot(1,2,1)
plt.plot(range(1, epochs+1), train_acc, label="Train Accuracy", marker='o')
plt.plot(range(1, epochs+1), val_acc, label="Validation Accuracy", marker='s')
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy (MLP)")
plt.legend()
plt.grid(True)

# Plot Loss
plt.subplot(1,2,2)
plt.plot(range(1, epochs+1), train_loss, label="Training Loss", color='red', marker='x')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss (MLP)")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Data akurasi model (dari hasil penelitian Anda)
models = ['Random Forest', 'MLP', 'K-NN', 'SVM']
accuracy = [99.63, 96.84, 92.10, 89.13]  # dalam persen

# Membuat bar chart
plt.figure(figsize=(8, 5))
bars = plt.bar(models, accuracy, color=['#4CAF50', '#2196F3', '#FFC107', '#F44336'])

# Menambahkan label akurasi di atas setiap batang
for bar, acc in zip(bars, accuracy):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{acc}%', ha='center', fontsize=10, fontweight='bold')

# Judul dan label sumbu
plt.ylabel('Accuracy (%)')
plt.ylim(0, 105)  # batas atas sedikit lebih dari 100 untuk memberi ruang teks

# Tampilkan chart
plt.show()


In [ ]:
import joblib

best_model_name = max(results, key=lambda k: results[k]['accuracy'])
best_model = results[best_model_name]['model']

joblib.dump(best_model, f'{best_model_name}_model.pkl')
print(f"📦 Model terbaik '{best_model_name}' disimpan sebagai '{best_model_name}_model.pkl'")


In [ ]:
import pandas as pd

df = pd.read_csv('dataset_keypoints_final.csv')

print(df.columns.tolist())

print(df.head(3))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score)
from sklearn.pipeline import Pipeline

from scipy.stats import wilcoxon
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv1D, MaxPooling1D, LSTM, Dense,
                                     Dropout, GlobalAveragePooling1D)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

print(f'TensorFlow version : {tf.__version__}')
print(f'NumPy version      : {np.__version__}')

In [ ]:
# ── Konfigurasi global ──────────────────────────────────────
CSV_PATH         = 'dataset_keypoints_final.csv'
FRAMES_PER_VIDEO = 30        # sesuai paper: 30 frame per video
N_FOLDS          = 5
RANDOM_STATE     = 42
SPLIT_CONFIGS    = {
    '80:20': 0.20,
    '70:30': 0.30,
}

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f'Shape awal        : {df.shape}')
print(f'Jumlah video unik : {df["video"].nunique()}')
print(f'Distribusi kelas  :\n{df["class"].value_counts()}')

kpt_cols = [c for c in df.columns if c.startswith('kpt_')]
print(f'\nJumlah kolom keypoint: {len(kpt_cols)}')
df.head(3)

In [ ]:
X_frame = df[kpt_cols].values

le = LabelEncoder()
y_frame = le.fit_transform(df['class'].values)
class_names = le.classes_

print(f'Kelas   : {list(class_names)}')
print(f'X shape : {X_frame.shape}')
print(f'y shape : {y_frame.shape}')

In [ ]:
def compute_delta_features(df, kpt_cols):
    """
    Hitung delta dan delta-delta per video.
    Output: (n_frames, n_kpt * 3) → original + delta + delta-delta
    """
    videos = df['video'].values
    X = df[kpt_cols].fillna(0).values.copy()

    delta       = np.zeros_like(X)
    delta_delta = np.zeros_like(X)

    for vid in df['video'].unique():
        idx = np.where(videos == vid)[0]
        for i, pos in enumerate(idx):
            if i == 0:
                delta[pos]       = 0.0
                delta_delta[pos] = 0.0
            else:
                delta[pos] = X[pos] - X[idx[i-1]]
                delta_delta[pos] = 0.0 if i == 1 else delta[pos] - delta[idx[i-1]]

    X_out = np.concatenate([X, delta, delta_delta], axis=1)
    print(f'  Original : {X.shape[1]} fitur')
    print(f'  Delta    : {delta.shape[1]} fitur')
    print(f'  Delta-Δ  : {delta_delta.shape[1]} fitur')
    print(f'  Total    : {X_out.shape[1]} fitur')
    return X_out

X_delta = compute_delta_features(df, kpt_cols)

In [ ]:
def build_video_sequences(df, kpt_cols, frames_per_video=30):
    """
    Susun data menjadi (n_videos, frames_per_video, n_features).
    """
    imputer = SimpleImputer(strategy='constant', fill_value=0.0)
    scaler  = StandardScaler()

    X_imp   = imputer.fit_transform(df[kpt_cols].values)
    videos  = df['video'].values
    classes = df['class'].values

    sequences, labels = [], []
    for vid in df['video'].unique():
        idx    = np.where(videos == vid)[0]
        frames = X_imp[idx]
        # truncate atau pad ke frames_per_video
        if len(frames) >= frames_per_video:
            frames = frames[:frames_per_video]
        else:
            pad    = np.zeros((frames_per_video - len(frames), frames.shape[1]))
            frames = np.vstack([frames, pad])
        sequences.append(frames)
        labels.append(classes[idx[0]])

    X_seq  = np.array(sequences)          # (n_videos, 30, n_feat)
    y_seq  = le.transform(labels)

    # Normalisasi per fitur
    n_vid, n_fr, n_feat = X_seq.shape
    X_flat = scaler.fit_transform(X_seq.reshape(-1, n_feat))
    X_seq  = X_flat.reshape(n_vid, n_fr, n_feat)

    print(f'  Jumlah video   : {X_seq.shape[0]}')
    print(f'  Sequence shape : {X_seq.shape}  → (videos, frames, features)')
    return X_seq, y_seq

X_seq, y_seq = build_video_sequences(df, kpt_cols, FRAMES_PER_VIDEO)
n_classes   = len(class_names)
n_timesteps = X_seq.shape[1]
n_features  = X_seq.shape[2]

In [ ]:
# ── Model Klasik ────────────────────────────────────────────
def make_classical_pipeline(model_name, use_scaler=True):
    steps = [('imputer', SimpleImputer(strategy='constant', fill_value=0.0))]
    if use_scaler:
        steps.append(('scaler', StandardScaler()))
    if model_name == 'RF':
        steps.append(('clf', RandomForestClassifier(
            n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)))
    elif model_name == 'MLP':
        steps.append(('clf', MLPClassifier(
            hidden_layer_sizes=(256, 128), activation='relu',
            max_iter=500, random_state=RANDOM_STATE)))
    elif model_name == 'SVM':
        steps.append(('clf', SVC(kernel='rbf', random_state=RANDOM_STATE)))
    elif model_name == 'KNN':
        steps.append(('clf', KNeighborsClassifier(n_neighbors=5)))
    return Pipeline(steps)


# ── Model Deep Learning ─────────────────────────────────────
def build_cnn1d(n_timesteps, n_features, n_classes):
    model = Sequential([
        Conv1D(64, kernel_size=3, activation='relu',
               input_shape=(n_timesteps, n_features)),
        MaxPooling1D(pool_size=2),
        Dropout(0.3),
        Conv1D(128, kernel_size=3, activation='relu'),
        GlobalAveragePooling1D(),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(n_classes, activation='softmax')
    ])
    model.compile(optimizer=Adam(1e-3),
                  loss='categorical_crossentropy', metrics=['accuracy'])
    return model


def build_lstm(n_timesteps, n_features, n_classes):
    model = Sequential([
        LSTM(128, input_shape=(n_timesteps, n_features)),
        Dropout(0.3),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(n_classes, activation='softmax')
    ])
    model.compile(optimizer=Adam(1e-3),
                  loss='categorical_crossentropy', metrics=['accuracy'])
    return model


es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
print('Definisi model selesai.')

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
cv_results = {}   # model_name → list akurasi tiap fold

# Konfigurasi model klasik
classical_configs = {
    'RF (frame)'  : (make_classical_pipeline('RF',  use_scaler=False), X_frame, y_frame),
    'MLP (frame)' : (make_classical_pipeline('MLP', use_scaler=True),  X_frame, y_frame),
    'SVM (frame)' : (make_classical_pipeline('SVM', use_scaler=True),  X_frame, y_frame),
    'KNN (frame)' : (make_classical_pipeline('KNN', use_scaler=True),  X_frame, y_frame),
}
classical_delta_configs = {
    'RF (delta)'  : (make_classical_pipeline('RF',  use_scaler=False), X_delta, y_frame),
    'MLP (delta)' : (make_classical_pipeline('MLP', use_scaler=True),  X_delta, y_frame),
    'SVM (delta)' : (make_classical_pipeline('SVM', use_scaler=True),  X_delta, y_frame),
    'KNN (delta)' : (make_classical_pipeline('KNN', use_scaler=True),  X_delta, y_frame),
}

# ── Jalankan model klasik ────────────────────────────────────
print('=== Model Klasik ===')
for model_name, (pipe, X, y) in {**classical_configs, **classical_delta_configs}.items():
    fold_accs = []
    for tr_idx, te_idx in skf.split(X, y):
        pipe.fit(X[tr_idx], y[tr_idx])
        fold_accs.append(accuracy_score(y[te_idx], pipe.predict(X[te_idx])) * 100)
    cv_results[model_name] = fold_accs
    print(f'  {model_name:20s} | mean={np.mean(fold_accs):.2f}%  std=±{np.std(fold_accs):.2f}%')

# ── Jalankan DL ─────────────────────────────────────────────
print('\n=== Deep Learning Baselines ===')
dl_configs = {'1D-CNN': build_cnn1d, 'LSTM': build_lstm}

for model_name, build_fn in dl_configs.items():
    fold_accs = []
    for fold_i, (tr_idx, te_idx) in enumerate(skf.split(X_seq, y_seq)):
        model = build_fn(n_timesteps, n_features, n_classes)
        model.fit(
            X_seq[tr_idx], to_categorical(y_seq[tr_idx], n_classes),
            validation_split=0.15, epochs=100, batch_size=16,
            callbacks=[es], verbose=0
        )
        _, acc = model.evaluate(
            X_seq[te_idx], to_categorical(y_seq[te_idx], n_classes), verbose=0)
        fold_accs.append(acc * 100)
        tf.keras.backend.clear_session()
        print(f'    Fold {fold_i+1}/5 selesai — acc={acc*100:.2f}%')
    cv_results[model_name] = fold_accs
    print(f'  {model_name:20s} | mean={np.mean(fold_accs):.2f}%  std=±{np.std(fold_accs):.2f}%')

In [ ]:
rows = []
for model_name, accs in cv_results.items():
    row = {'Model': model_name}
    for i, a in enumerate(accs):
        row[f'F{i+1}(%)'] = round(a, 2)
    row['Mean(%)'] = round(np.mean(accs), 2)
    row['Std(%)']  = round(np.std(accs), 2)
    rows.append(row)

df_summary = pd.DataFrame(rows)
df_summary.to_csv('cv_results_extended.csv', index=False)
print('Disimpan ke cv_results_extended.csv')
df_summary

In [ ]:
def cohens_d(a, b):
    """Cohen's d untuk dua sampel berpasangan."""
    diff = np.array(a) - np.array(b)
    return np.mean(diff) / (np.std(diff, ddof=1) + 1e-10)

baseline      = 'RF (frame)'
baseline_accs = cv_results[baseline]
stat_rows     = []

for model_name, accs in cv_results.items():
    if model_name == baseline:
        continue
    try:
        _, p = wilcoxon(baseline_accs, accs)
    except Exception:
        p = np.nan
    d   = cohens_d(baseline_accs, accs)
    sig = 'significant' if (p is not np.nan and p < 0.05) else 'not significant'
    eff = 'large' if abs(d) >= 0.8 else ('medium' if abs(d) >= 0.5 else 'small')
    stat_rows.append({
        'Comparison'   : f'{baseline} vs {model_name}',
        'p-value'      : round(p, 4) if p is not np.nan else np.nan,
        'Significance' : sig,
        "Cohen's d"    : round(d, 3),
        'Effect size'  : eff
    })

df_stat = pd.DataFrame(stat_rows)
df_stat.to_csv('statistical_test_extended.csv', index=False)
print('Disimpan ke statistical_test_extended.csv')
df_stat

In [ ]:
split_rows = []

for split_name, test_size in SPLIT_CONFIGS.items():
    print(f'\n── Split {split_name} ──')

    # Model klasik: frame
    for model_name, (pipe, X, y) in classical_configs.items():
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=test_size, stratify=y, random_state=RANDOM_STATE)
        pipe.fit(X_tr, y_tr)
        y_pred = pipe.predict(X_te)
        split_rows.append({
            'Split': split_name, 'Model': model_name,
            'Accuracy(%)' : round(accuracy_score(y_te, y_pred)*100, 2),
            'F1-macro(%)' : round(f1_score(y_te, y_pred, average='macro')*100, 2)
        })

    # Model klasik: delta
    for model_name, (pipe, X, y) in classical_delta_configs.items():
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=test_size, stratify=y, random_state=RANDOM_STATE)
        pipe.fit(X_tr, y_tr)
        y_pred = pipe.predict(X_te)
        split_rows.append({
            'Split': split_name, 'Model': model_name,
            'Accuracy(%)' : round(accuracy_score(y_te, y_pred)*100, 2),
            'F1-macro(%)' : round(f1_score(y_te, y_pred, average='macro')*100, 2)
        })

    # DL: video-level
    for model_name, build_fn in dl_configs.items():
        X_tr, X_te, y_tr, y_te = train_test_split(
            X_seq, y_seq, test_size=test_size, stratify=y_seq, random_state=RANDOM_STATE)
        model = build_fn(n_timesteps, n_features, n_classes)
        model.fit(
            X_tr, to_categorical(y_tr, n_classes),
            validation_split=0.15, epochs=100, batch_size=16,
            callbacks=[es], verbose=0
        )
        y_pred = np.argmax(model.predict(X_te, verbose=0), axis=1)
        split_rows.append({
            'Split': split_name, 'Model': model_name,
            'Accuracy(%)' : round(accuracy_score(y_te, y_pred)*100, 2),
            'F1-macro(%)' : round(f1_score(y_te, y_pred, average='macro')*100, 2)
        })
        tf.keras.backend.clear_session()
        print(f'  {model_name} ({split_name}) selesai')

df_splits = pd.DataFrame(split_rows)
df_splits.to_csv('split_results_extended.csv', index=False)
print('\nDisimpan ke split_results_extended.csv')
df_splits

In [ ]:
rf_delta_pipe = make_classical_pipeline('RF', use_scaler=False)
X_tr, X_te, y_tr, y_te = train_test_split(
    X_delta, y_frame, test_size=0.30,
    stratify=y_frame, random_state=RANDOM_STATE)
rf_delta_pipe.fit(X_tr, y_tr)
y_pred_rf = rf_delta_pipe.predict(X_te)

cm = confusion_matrix(y_te, y_pred_rf)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_title('Confusion Matrix — RF + Delta Features (70:30 split)')
ax.set_ylabel('Actual')
ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig('confusion_matrix_RF_delta_7030.png', dpi=150)
plt.show()

total_err = np.sum(cm) - np.trace(cm)
err_rate  = total_err / len(y_te) * 100
print(f'Total error: {total_err} / {len(y_te)} = {err_rate:.2f}%')
print('\nClassification Report:')
print(classification_report(y_te, y_pred_rf, target_names=class_names))

In [ ]:
model_order = [
    'RF (frame)', 'RF (delta)',
    'MLP (frame)', 'MLP (delta)',
    'KNN (frame)', 'KNN (delta)',
    'SVM (frame)', 'SVM (delta)',
    '1D-CNN', 'LSTM'
]
names = [m for m in model_order if m in cv_results]
means = [np.mean(cv_results[m]) for m in names]
stds  = [np.std(cv_results[m])  for m in names]

color_map = {'RF': '#2196F3', 'MLP': '#4CAF50', 'KNN': '#FF9800',
             'SVM': '#F44336', '1D-CNN': '#9C27B0', 'LSTM': '#795548'}
colors = [next(v for k, v in color_map.items() if k in n) for n in names]

fig, ax = plt.subplots(figsize=(13, 5))
bars = ax.bar(names, means, yerr=stds, capsize=4,
              color=colors, alpha=0.85, edgecolor='white')
ax.set_ylim(55, 108)
ax.set_ylabel('Mean Accuracy (%)')
ax.set_title('Perbandingan Akurasi 5-Fold CV\nFrame-wise vs Delta Features vs Deep Learning Baselines')
ax.axhline(y=90, color='gray', linestyle='--', linewidth=0.8, alpha=0.6, label='90% threshold')
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{mean:.1f}%', ha='center', va='bottom', fontsize=8)
ax.legend(fontsize=9)
plt.xticks(rotation=30, ha='right', fontsize=9)
plt.tight_layout()
plt.savefig('model_comparison_extended.png', dpi=150)
plt.show()
print('Disimpan ke model_comparison_extended.png')

In [ ]:
import pandas as pd
import os

candidates = [
    "dataset_keypoints.csv",
    "dataset_keypoints_raw.csv",
    "dataset_keypoints_clean.csv",
    "dataset_keypoints_final.csv",
    "dataset_keypoints_finals.csv",
    "dataset_keypoints_fixed.csv",
]

for f in candidates:
    if os.path.exists(f):
        try:
            tmp = pd.read_csv(f)
            n_video = tmp['video'].nunique() if 'video' in tmp.columns else '-'
            print(f"{f:35s} shape={tmp.shape}  video_unik={n_video}")
        except Exception as e:
            print(f"{f:35s} GAGAL DIBACA: {e}")
    else:
        print(f"{f:35s} (file tidak ada di folder ini)")

In [ ]:
import pandas as pd
clean = pd.read_csv("dataset_keypoints_clean.csv")
print("Kolom:", clean.columns.tolist()[:5], "...", clean.columns.tolist()[-3:])
print("Jumlah kolom:", len(clean.columns))
label_col = clean.columns[-1]
print(f"\nDistribusi kelas (kolom '{label_col}'):")
print(clean[label_col].value_counts())

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer

clean = pd.read_csv("dataset_keypoints_clean.csv")
feat_cols = [c for c in clean.columns if c != "label"]

# Imputasi zero dulu (paling sederhana, sesuai Section 3.4)
X_check = SimpleImputer(strategy="constant", fill_value=0.0).fit_transform(clean[feat_cols])
le_check = LabelEncoder()
y_check = le_check.fit_transform(clean["label"])

skf_check = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(
    RandomForestClassifier(n_estimators=100, random_state=42),
    X_check, y_check, cv=skf_check, scoring="accuracy"
)
print(f"RF 5-fold CV accuracy: {scores.mean()*100:.2f}% (+/-{scores.std()*100:.2f}%)")
print("Target dari paper: 98.28% (+/-0.58%)")

In [ ]:
clean2 = pd.read_csv("dataset_keypoints_clean.csv")
feat_cols = [c for c in clean2.columns if c != "label"]

for label in clean2["label"].unique():
    mask = clean2["label"] == label
    col_means = clean2.loc[mask, feat_cols].replace(0, np.nan).mean(skipna=True)
    clean2.loc[mask, feat_cols] = clean2.loc[mask, feat_cols].replace(0, np.nan).fillna(col_means)

X_check2 = clean2[feat_cols].values
y_check2 = le_check.fit_transform(clean2["label"])

scores2 = cross_val_score(
    RandomForestClassifier(n_estimators=100, random_state=42),
    X_check2, y_check2, cv=skf_check, scoring="accuracy"
)
print(f"RF 5-fold CV (mean-impute): {scores2.mean()*100:.2f}% (+/-{scores2.std()*100:.2f}%)")

In [ ]:
raw = pd.read_csv("dataset_keypoints.csv")
print("Kolom dataset_keypoints.csv:")
print(raw.columns.tolist())

In [ ]:
import os
print(os.path.exists("frames"))
if os.path.exists("frames"):
    for class_folder in os.listdir("frames")[:1]:
        sample_dir = os.path.join("frames", class_folder)
        print(class_folder, "->", os.listdir(sample_dir)[:3])

In [ ]:
import cv2
import mediapipe as mp
import os
import pandas as pd
import numpy as np

mp_pose = mp.solutions.pose
mp_hands = mp.solutions.hands
mp_face = mp.solutions.face_mesh
pose = mp_pose.Pose(static_image_mode=True)
hands = mp_hands.Hands(static_image_mode=True, max_num_hands=2)
face = mp_face.FaceMesh(static_image_mode=True)

def extract_landmarks(image_path):
    image = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results_pose = pose.process(image_rgb)
    results_hands = hands.process(image_rgb)
    results_face = face.process(image_rgb)
    features = []
    if results_pose.pose_landmarks:
        for lm in results_pose.pose_landmarks.landmark:
            features.extend([lm.x, lm.y, lm.z])
    else:
        features.extend([0]*99)
    if results_hands.multi_hand_landmarks:
        for hand in results_hands.multi_hand_landmarks:
            for lm in hand.landmark:
                features.extend([lm.x, lm.y, lm.z])
        if len(results_hands.multi_hand_landmarks) == 1:
            features.extend([0]*63)
    else:
        features.extend([0]*126)
    if results_face.multi_face_landmarks:
        for lm in results_face.multi_face_landmarks[0].landmark[:20]:
            features.extend([lm.x, lm.y, lm.z])
    else:
        features.extend([0]*60)
    return features

frames_dir = "frames"
data = []

for class_folder in os.listdir(frames_dir):
    class_path = os.path.join(frames_dir, class_folder)
    if not os.path.isdir(class_path):
        continue
    print(f"Memproses kelas: {class_folder}")
    for img_file in os.listdir(class_path):
        img_path = os.path.join(class_path, img_file)
        if not (os.path.isfile(img_path) and img_file.endswith(".jpg")):
            continue
        video_name = img_file.rsplit("_frame", 1)[0]  # buang suffix "_frame000.jpg"
        features = extract_landmarks(img_path)
        features.append(class_folder)   # label
        features.append(video_name)     # video (utk signer_id)

In [ ]:
import os

# Cek 1: apakah variabel df_raw_sign masih ada di memori kernel
try:
    print("df_raw_sign ADA. Shape:", df_raw_sign.shape)
except NameError:
    print("df_raw_sign TIDAK ADA di memori -- proses belum selesai / belum sempat assign.")

# Cek 2: apakah file output sudah tersimpan di disk
if os.path.exists("dataset_keypoints_with_video.csv"):
    size_mb = os.path.getsize("dataset_keypoints_with_video.csv") / (1024*1024)
    print(f"File dataset_keypoints_with_video.csv ADA, ukuran {size_mb:.1f} MB")
else:
    print("File dataset_keypoints_with_video.csv BELUM ADA -- proses sebelumnya belum selesai.")

In [ ]:
import cv2
import mediapipe as mp
import os
import pandas as pd
import time

mp_pose = mp.solutions.pose
mp_hands = mp.solutions.hands
mp_face = mp.solutions.face_mesh
pose = mp_pose.Pose(static_image_mode=True)
hands = mp_hands.Hands(static_image_mode=True, max_num_hands=2)
face = mp_face.FaceMesh(static_image_mode=True)

def extract_landmarks(image_path):
    image = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results_pose = pose.process(image_rgb)
    results_hands = hands.process(image_rgb)
    results_face = face.process(image_rgb)
    features = []
    if results_pose.pose_landmarks:
        for lm in results_pose.pose_landmarks.landmark:
            features.extend([lm.x, lm.y, lm.z])
    else:
        features.extend([0]*99)
    if results_hands.multi_hand_landmarks:
        for hand in results_hands.multi_hand_landmarks:
            for lm in hand.landmark:
                features.extend([lm.x, lm.y, lm.z])
        if len(results_hands.multi_hand_landmarks) == 1:
            features.extend([0]*63)
    else:
        features.extend([0]*126)
    if results_face.multi_face_landmarks:
        for lm in results_face.multi_face_landmarks[0].landmark[:20]:
            features.extend([lm.x, lm.y, lm.z])
    else:
        features.extend([0]*60)
    return features

def process_class(class_folder, frames_dir="frames", progress_every=50):
    class_path = os.path.join(frames_dir, class_folder)
    data = []
    t0 = time.time()
    files = [f for f in os.listdir(class_path)
             if os.path.isfile(os.path.join(class_path, f)) and f.endswith(".jpg")]
    print(f"[{class_folder}] total gambar: {len(files)}")
    for i, img_file in enumerate(files):
        img_path = os.path.join(class_path, img_file)
        video_name = img_file.rsplit("_frame", 1)[0]
        features = extract_landmarks(img_path)
        features.append(class_folder)
        features.append(video_name)
        data.append(features)
        if (i + 1) % progress_every == 0:
            elapsed = time.time() - t0
            print(f"  [{class_folder}] {i+1}/{len(files)} selesai ({elapsed:.0f}s)")
    n_feat = len(data[0]) - 2
    columns = [f"f{i}" for i in range(n_feat)] + ["label", "video"]
    df_class = pd.DataFrame(data, columns=columns)
    out_name = f"keypoints_{class_folder.replace(' ', '_')}.csv"
    df_class.to_csv(out_name, index=False)
    print(f"[{class_folder}] SELESAI, disimpan -> {out_name}  shape={df_class.shape}")
    return df_class

In [ ]:
process_class("Halo")

In [ ]:
process_class("Apa Kabar")

In [ ]:
process_class("Maaf")

In [ ]:
process_class("Sama-sama")

In [ ]:
process_class("Terima kasih")

In [ ]:
import os

print("Isi folder 'frames':")
for name in sorted(os.listdir("frames")):
    full_path = os.path.join("frames", name)
    if os.path.isdir(full_path):
        jpgs = [f for f in os.listdir(full_path) if f.endswith(".jpg")]
        print(f"  '{name}'  ->  {len(jpgs)} file .jpg")
    else:
        print(f"  '{name}'  -> (ini FILE, bukan folder)")

In [ ]:
data_dir = r"path/to/Dataset Isyarat"  # sesuaikan kalau lokasi di laptop Anda beda

import os
if not os.path.exists(data_dir):
    print("Folder TIDAK DITEMUKAN di path ini. Cari lokasi yang benar dulu.")
else:
    print(f"Folder ditemukan: {data_dir}\n")
    for class_folder in sorted(os.listdir(data_dir)):
        class_path = os.path.join(data_dir, class_folder)
        if os.path.isdir(class_path):
            videos = [f for f in os.listdir(class_path) if f.endswith(".mp4")]
            print(f"  '{class_folder}'  ->  {len(videos)} video .mp4")

In [ ]:
import os

class_path = os.path.join(data_dir, "Apa Kabar")
videos = [f for f in os.listdir(class_path) if f.endswith(".mp4")]

# Ambil nama signer = bagian sebelum underscore pertama
signers = sorted(set(v.split("_", 1)[0] for v in videos))
print(f"Jumlah signer terdeteksi: {len(signers)}")
for s in signers:
    count = sum(1 for v in videos if v.startswith(s + "_"))
    print(f"  '{s}'  ->  {count} video")

In [ ]:
import cv2
import mediapipe as mp
import os
import time
import numpy as np
import pandas as pd

data_dir = r"path/to/Dataset Isyarat"
SIGNERS_PAPER = ["S1", "S2", "S3", "S4", "S5", "S6", "S7"]
N_FRAMES = 30

pose = mp.solutions.pose.Pose(static_image_mode=True)
hands = mp.solutions.hands.Hands(static_image_mode=True, max_num_hands=2)
face = mp.solutions.face_mesh.FaceMesh(static_image_mode=True)

def landmarks_from_frame(frame_bgr):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    r_pose, r_hands, r_face = pose.process(rgb), hands.process(rgb), face.process(rgb)
    f = []
    if r_pose.pose_landmarks:
        for lm in r_pose.pose_landmarks.landmark:
            f.extend([lm.x, lm.y, lm.z])
    else:
        f.extend([0]*99)
    if r_hands.multi_hand_landmarks:
        for h in r_hands.multi_hand_landmarks:
            for lm in h.landmark:
                f.extend([lm.x, lm.y, lm.z])
        if len(r_hands.multi_hand_landmarks) == 1:
            f.extend([0]*63)
    else:
        f.extend([0]*126)
    if r_face.multi_face_landmarks:
        for lm in r_face.multi_face_landmarks[0].landmark[:20]:
            f.extend([lm.x, lm.y, lm.z])
    else:
        f.extend([0]*60)
    return f

def process_class_video(class_folder):
    class_path = os.path.join(data_dir, class_folder)
    videos = sorted(v for v in os.listdir(class_path)
                    if v.endswith(".mp4") and v.split("_", 1)[0] in SIGNERS_PAPER)
    print(f"[{class_folder}] video terpakai: {len(videos)} (target 35)")
    data, t0 = [], time.time()
    for i, v in enumerate(videos):
        cap = cv2.VideoCapture(os.path.join(class_path, v))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        idxs = np.linspace(0, total - 1, N_FRAMES).astype(int)
        got = 0
        for idx in idxs:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
            ok, frame = cap.read()
            if not ok:
                continue
            feats = landmarks_from_frame(frame)
            feats += [class_folder, os.path.splitext(v)[0]]
            data.append(feats)
            got += 1
        cap.release()
        if got != N_FRAMES:
            print(f"  PERINGATAN: {v} hanya dapat {got}/{N_FRAMES} frame")
        print(f"  {i+1}/{len(videos)} {v} ({time.time()-t0:.0f}s)")
    n_feat = len(data[0]) - 2
    cols = [f"f{i}" for i in range(n_feat)] + ["label", "video"]
    df = pd.DataFrame(data, columns=cols)
    out = f"kp210_{class_folder.replace(' ', '_')}.csv"
    df.to_csv(out, index=False)
    print(f"[{class_folder}] SELESAI -> {out} shape={df.shape} (target (1050, 287))")
    return df

In [ ]:
process_class_video("Apa Kabar")

In [ ]:
process_class_video("Halo")

In [ ]:
process_class_video("Maaf")

In [ ]:
process_class_video("Sama-sama")

In [ ]:
process_class_video("Terima kasih")

In [ ]:
process_class_video("Tolong")

In [ ]:
import glob
files = sorted(glob.glob("kp210_*.csv"))
print(files)
df_raw_sign = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
df_raw_sign.to_csv("dataset_keypoints_with_video.csv", index=False)
print(df_raw_sign.shape, "(target: (6300, 287))")
print(df_raw_sign["label"].value_counts())
print(df_raw_sign["video"].nunique(), "video unik (target 210)")

In [ ]:
counts = df_raw_sign.groupby("video").size()
print(counts[counts != 30])

In [ ]:
cap = cv2.VideoCapture(os.path.join(data_dir, "Halo", "S4_Halo4.mp4"))
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print("Frame menurut metadata:", total)

idxs = np.linspace(0, total - 1, N_FRAMES).astype(int)
for idx in idxs:
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
    ok, _ = cap.read()
    if not ok:
        print("Gagal dibaca di index:", idx)
cap.release()

In [ ]:
        last_feats = None
        got = 0
        for idx in idxs:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
            ok, frame = cap.read()
            if ok:
                feats = landmarks_from_frame(frame)
                last_feats = feats
            elif last_feats is not None:
                feats = last_feats
            else:
                continue
            data.append(feats + [class_folder, os.path.splitext(v)[0]])
            got += 1

In [ ]:
process_class_video("Halo")

In [ ]:
def process_class_video(class_folder):
    class_path = os.path.join(data_dir, class_folder)
    videos = sorted(v for v in os.listdir(class_path)
                    if v.endswith(".mp4") and v.split("_", 1)[0] in SIGNERS_PAPER)
    print(f"[{class_folder}] video terpakai: {len(videos)} (target 35)")
    data, t0 = [], time.time()
    for i, v in enumerate(videos):
        cap = cv2.VideoCapture(os.path.join(class_path, v))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        idxs = np.linspace(0, total - 1, N_FRAMES).astype(int)
        last_feats = None
        got = 0
        for idx in idxs:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
            ok, frame = cap.read()
            if ok:
                feats = landmarks_from_frame(frame)
                last_feats = feats
            elif last_feats is not None:
                feats = last_feats
                print(f"  CATATAN: {v} frame idx {idx} gagal dibaca, diisi dari frame valid terakhir")
            else:
                continue
            data.append(feats + [class_folder, os.path.splitext(v)[0]])
            got += 1
        cap.release()
        if got != N_FRAMES:
            print(f"  PERINGATAN: {v} hanya dapat {got}/{N_FRAMES} frame")
        print(f"  {i+1}/{len(videos)} {v} ({time.time()-t0:.0f}s)")
    n_feat = len(data[0]) - 2
    cols = [f"f{i}" for i in range(n_feat)] + ["label", "video"]
    df = pd.DataFrame(data, columns=cols)
    out = f"kp210_{class_folder.replace(' ', '_')}.csv"
    df.to_csv(out, index=False)
    print(f"[{class_folder}] SELESAI -> {out} shape={df.shape} (target (1050, 287))")
    return df

In [ ]:
process_class_video("Halo")

In [ ]:
import glob
files = sorted(glob.glob("kp210_*.csv"))
print(files)
df_raw_sign = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
df_raw_sign.to_csv("dataset_keypoints_with_video.csv", index=False)

print(df_raw_sign.shape, "(target: (6300, 287))")
print(df_raw_sign["label"].value_counts())
print(df_raw_sign["video"].nunique(), "video unik (target 210)")

counts = df_raw_sign.groupby("video").size()
print("Video dengan jumlah frame != 30:", counts[counts != 30].to_dict())

In [ ]:
print(CSV_PATH)
print(df.shape)
print(len(kpt_cols), "kolom fitur")
print(kpt_cols[:3], "...", kpt_cols[-3:])
print(df["video"].nunique(), "video")
print(df.groupby("class").size())

In [ ]:
import os, glob, time
for f in sorted(glob.glob("**/*.csv", recursive=True)):
    s = os.stat(f)
    print(f"{f:60s} {s.st_size/1e6:8.1f} MB  {time.ctime(s.st_mtime)}")

In [ ]:
import pandas as pd

for f in ["dataset_keypoints_finals.csv", "dataset_keypoints_raw.csv", "dataset_keypoints_clean.csv"]:
    d = pd.read_csv(f)
    print("=" * 60)
    print(f, d.shape)
    print("kolom awal :", d.columns[:3].tolist(), "| kolom akhir:", d.columns[-4:].tolist())
    if "video" in d.columns:
        print("video unik :", d["video"].nunique())
    lab = "class" if "class" in d.columns else d.columns[-1]
    print(d[lab].value_counts().to_dict())

In [ ]:
import shutil
for f in ["dataset_keypoints_clean.csv", "dataset_keypoints_finals.csv",
          "dataset_keypoints_raw.csv", "dataset_keypoints_final.csv"]:
    shutil.copy(f, "BACKUP_" + f)
print("aman")

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

d = pd.read_csv("dataset_keypoints_clean.csv")
fc = [c for c in d.columns if c.startswith("f")]
X, y = d[fc].values, d["label"].values

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE)
pipe = make_classical_pipeline("RF", use_scaler=False)
pipe.fit(X_tr, y_tr)
print("RF 70:30 acc:", accuracy_score(y_te, pipe.predict(X_te)))

In [ ]:
import numpy as np, pandas as pd

SIGNERS_PAPER = ["S1", "S2", "S3", "S4", "S5", "S6", "S7"]
raw = pd.read_csv("dataset_keypoints_raw.csv")
print(raw["video"].unique()[:5])   # lihat format nama video
raw["signer"] = raw["video"].str.split("_", n=1).str[0]
print(raw["signer"].value_counts())

paper = raw[raw["signer"].isin(SIGNERS_PAPER)].copy()
print("subset paper:", paper.shape, "| video:", paper["video"].nunique())

kc = [c for c in paper.columns if c.startswith("kpt_")]
r = paper[kc].values.astype(float).reshape(len(paper), 107, 3)
missing = (np.isnan(r).all(axis=2) | (r == 0).all(axis=2)).sum(axis=1)
print(pd.Series(missing).describe())
print("dibuang (>30):", (missing > 30).sum(), "| sisa:", (missing <= 30).sum())

In [ ]:
import numpy as np, pandas as pd

raw = pd.read_csv("dataset_keypoints_raw.csv")
raw["signer"] = raw["video"].str.split("_", n=1).str[0]
paper = raw[raw["signer"].isin(["S1","S2","S3","S4","S5","S6","S7"])].copy()
kc = [c for c in paper.columns if c.startswith("kpt_")]

# 1. titik mana yang selalu kosong?
vals = paper[kc].astype(float)
always_empty = [c for c in kc if (vals[c].isna() | (vals[c] == 0)).all()]
print("kolom selalu kosong:", len(always_empty), always_empty[:6], "...", always_empty[-3:])

# 2. apakah clean.csv adalah subset dari data mentah ini?
clean = pd.read_csv("dataset_keypoints_clean.csv")
fc = [c for c in clean.columns if c.startswith("f")]       # 285 fitur
cols285 = kc[:225] + kc[225:285]                            # pose + tangan + 20 titik wajah pertama

def key(df, cols):
    return df[cols].fillna(0).round(5).astype(str).agg("|".join, axis=1)

paper["_k"] = key(paper, cols285)
clean["_k"] = key(clean.rename(columns=dict(zip(fc, cols285))), cols285)
m = clean[["_k"]].merge(paper[["_k", "video", "class"]], on="_k", how="left")
print("baris clean.csv yang ketemu di data mentah:", m["video"].notna().sum(), "dari", len(clean))
print("video unik ter-match:", m["video"].nunique())

In [ ]:
import numpy as np, pandas as pd

clean = pd.read_csv("dataset_keypoints_clean.csv")
fc = [c for c in clean.columns if c.startswith("f")]
A = np.round(np.nan_to_num(clean[fc].values.astype(float)), 5)
clean_keys = {hash(r.tobytes()): i for i, r in enumerate(A)}

for f in ["dataset_keypoints.csv", "dataset_keypoints_fixed.csv"]:
    d = pd.read_csv(f)
    print("=" * 60)
    print(f, d.shape)
    print("kolom awal:", d.columns[:3].tolist(), "| kolom akhir:", d.columns[-3:].tolist())
    num = d.select_dtypes("number").columns[:285]
    B = np.round(np.nan_to_num(d[num].values.astype(float)), 5)
    pos = [i for i, r in enumerate(B) if hash(r.tobytes()) in clean_keys]
    print("baris clean.csv ketemu di file ini:", len(pos), "dari", len(clean))
    if pos:
        print("indeks baris pertama/terakhir yang cocok:", pos[:5], pos[-3:])

In [ ]:
import numpy as np, pandas as pd
from scipy.spatial.distance import cdist

old = pd.read_csv("dataset_keypoints.csv")
fc = [c for c in old.columns if c.startswith("f")]

# 1. Apakah urutan baris per kelas, 1050 baris per blok?
runs = old["label"].ne(old["label"].shift()).cumsum().value_counts().sort_index().tolist()
print("panjang blok kelas:", runs)
print("urutan kelas:", old["label"].iloc[::1050].tolist())

# 2. Cocokkan ke data ber-video hasil ekstraksi kita (nearest neighbor per kelas)
new = pd.read_csv("dataset_keypoints_with_video.csv")
fcn = [c for c in new.columns if c.startswith("f")]
parts = []
for lab in old["label"].unique():
    o, n = old[old["label"] == lab], new[new["label"] == lab]
    D = cdist(np.nan_to_num(o[fc].values), np.nan_to_num(n[fcn].values))
    j = D.argmin(1)
    parts.append(pd.DataFrame({"video_nn": n["video"].values[j], "dist": D.min(1)}, index=o.index))
res = pd.concat(parts).sort_index()
print(res["dist"].describe())

res["blok"] = res.index // 30
konsisten = res.groupby("blok")["video_nn"].agg(lambda s: s.value_counts().iloc[0] / len(s))
print("blok 30 frame yang >=90% menunjuk ke satu video:", round((konsisten >= 0.9).mean(), 3))

In [ ]:
import numpy as np, pandas as pd

SIGNERS = ["S1", "S2", "S3", "S4", "S5", "S6", "S7"]
old = pd.read_csv("dataset_keypoints.csv")
clean = pd.read_csv("dataset_keypoints_clean.csv")
fc = [c for c in old.columns if c.startswith("f")]
A_old = np.round(np.nan_to_num(old[fc].values.astype(float)), 5)
A_cln = np.round(np.nan_to_num(clean[fc].values.astype(float)), 5)

# --- posisi -> video/signer ---
pos = old.index.values
blok_kelas, v = pos % 1050, (pos % 1050) // 30
old["signer"] = [SIGNERS[i // 5] for i in v]
old["video_no"] = v % 5 + 1
old["video"] = old["signer"] + "_" + old["label"].str.replace(" ", "") + old["video_no"].astype(str)

# --- cek A: lebar bahu per signer konsisten di semua kelas? ---
sw = np.hypot(A_old[:, 33] - A_old[:, 36], A_old[:, 34] - A_old[:, 37])  # landmark 11 vs 12
sw = np.where((A_old[:, 33] == 0) & (A_old[:, 36] == 0), np.nan, sw)
old["sw"] = sw
M = old.groupby(["label", "signer"])["sw"].median().unstack()[SIGNERS]
print(M.round(3))
cors = np.corrcoef(M.values)
print("korelasi pola signer antar kelas (min):", cors[np.triu_indices(6, 1)].min().round(3))
within = old.groupby(["label", "signer"])["sw"].std().mean()
between = M.std(axis=1).mean()
print(f"std dalam signer {within:.4f} vs antar signer {between:.4f}")

# --- cek B: lonjakan di batas video ---
d = np.linalg.norm(np.diff(A_old, axis=0), axis=1)
idx = np.arange(1, len(old))
batas = d[(idx % 30) == 0]
dalam = d[(idx % 30) != 0]
print(f"jarak frame berurutan: batas video {np.median(batas):.3f} vs dalam video {np.median(dalam):.3f}")

# --- clean.csv adalah subsequence dari old (urutan terjaga)? ---
j, map_idx = 0, []
for i in range(len(A_old)):
    if j < len(A_cln) and np.array_equal(A_old[i], A_cln[j]):
        map_idx.append(i); j += 1
print("baris clean yang terpetakan berurutan:", j, "dari", len(clean))
ok_label = j == len(clean) and (old.loc[map_idx, "label"].values == clean["label"].values).all()
print("label cocok:", ok_label)

if ok_label:
    out = clean.copy()
    out["signer"] = old.loc[map_idx, "signer"].values
    out["video"] = old.loc[map_idx, "video"].values
    out["old_row"] = map_idx
    out.to_csv("dataset_paper_5379_with_video.csv", index=False)
    print("Disimpan -> dataset_paper_5379_with_video.csv", out.shape)
    print(out["signer"].value_counts())
    print(out["video"].nunique(), "video unik")
else:
    print("Pemetaan urutan gagal, TIDAK menyimpan apa pun.")

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, LeaveOneGroupOut, cross_val_score
from sklearn.metrics import accuracy_score, f1_score

d = pd.read_csv("dataset_paper_5379_with_video.csv")
fc = [c for c in d.columns if c.startswith("f") and c[1:].isdigit()]
X, y = d[fc].values, d["label"].values
g_video, g_signer = d["video"].values, d["signer"].values
print(X.shape, len(fc), "fitur |", d["video"].nunique(), "video |", d["signer"].nunique(), "signer")

def rf():
    return make_classical_pipeline("RF", use_scaler=False)

# A. Protokol paper: split acak per frame
cvA = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
a = cross_val_score(rf(), X, y, cv=cvA, scoring="accuracy")
print(f"A. frame-level (protokol paper): {a.mean():.4f} +/- {a.std():.4f}")

# B. Split per video (frame dari satu video tidak terpecah)
cvB = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
b = cross_val_score(rf(), X, y, cv=cvB, groups=g_video, scoring="accuracy")
print(f"B. video-level grouped:          {b.mean():.4f} +/- {b.std():.4f}")

# C. Leave-one-signer-out
rows = []
for tr, te in LeaveOneGroupOut().split(X, y, groups=g_signer):
    m = rf().fit(X[tr], y[tr]); p = m.predict(X[te])
    rows.append({"signer": g_signer[te[0]], "n": len(te),
                 "acc": accuracy_score(y[te], p), "f1_macro": f1_score(y[te], p, average="macro")})
c = pd.DataFrame(rows)
print(c.round(4))
print(f"C. LOSO rata-rata: {c['acc'].mean():.4f} (std {c['acc'].std():.4f}) | min {c['acc'].min():.4f}")
c.to_csv("c1_loso_results.csv", index=False)

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import confusion_matrix

d = pd.read_csv("dataset_paper_5379_with_video.csv")
fc = [c for c in d.columns if c.startswith("f") and c[1:].isdigit()]
X, y, sg = d[fc].values, d["label"].values, d["signer"].values

# 1. S5 diprediksi sebagai apa?
tr, te = sg != "S5", sg == "S5"
m = make_classical_pipeline("RF", use_scaler=False).fit(X[tr], y[tr])
p = m.predict(X[te])
labs = sorted(set(y))
print(pd.DataFrame(confusion_matrix(y[te], p, labels=labs), index=labs, columns=labs))

# 2. Profil tangan per signer
s1 = (np.abs(X[:, 99:162]).sum(1) > 0)    # slot tangan 1 terisi
s2 = (np.abs(X[:, 162:225]).sum(1) > 0)   # slot tangan 2 terisi
prof = pd.DataFrame({"signer": sg, "slot1": s1, "slot2": s2,
                     "wrist1_x": np.where(s1, X[:, 99], np.nan),
                     "sh_mid_x": (X[:, 33] + X[:, 36]) / 2,
                     "sh_mid_y": (X[:, 34] + X[:, 37]) / 2})
print(prof.groupby("signer").mean().round(3))

In [ ]:
n = len(X)
R = X.reshape(n, 95, 3).copy()
miss = (R[:, :, 0] == 0) & (R[:, :, 1] == 0)
cx = (R[:, 11, 0] + R[:, 12, 0]) / 2
cy = (R[:, 11, 1] + R[:, 12, 1]) / 2
sw = np.hypot(R[:, 11, 0] - R[:, 12, 0], R[:, 11, 1] - R[:, 12, 1])
sw = np.where(sw > 1e-6, sw, np.nan)
R[:, :, 0] = (R[:, :, 0] - cx[:, None]) / sw[:, None]
R[:, :, 1] = (R[:, :, 1] - cy[:, None]) / sw[:, None]
R[miss] = 0
Xn = np.nan_to_num(R.reshape(n, 285))

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import accuracy_score
rows = []
for tr, te in LeaveOneGroupOut().split(Xn, y, groups=sg):
    p = make_classical_pipeline("RF", use_scaler=False).fit(Xn[tr], y[tr]).predict(Xn[te])
    rows.append((sg[te[0]], accuracy_score(y[te], p)))
r = pd.DataFrame(rows, columns=["signer", "acc"])
print(r.round(4)); print("LOSO (ternormalisasi):", r["acc"].mean().round(4))

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import accuracy_score

# 1. Slot tangan kedua terisi, per kelas: S5 vs signer lain
s2 = (np.abs(X[:, 162:225]).sum(1) > 0)
t = pd.DataFrame({"label": y, "uut": np.where(sg == "S5", "S5", "lainnya"), "slot2": s2})
print(t.groupby(["label", "uut"])["slot2"].mean().unstack().round(3))

# 2. LOSO tanpa slot tangan kedua (ternormalisasi)
keep = np.r_[0:162, 225:285]
rows = []
for tr, te in LeaveOneGroupOut().split(Xn, y, groups=sg):
    p = make_classical_pipeline("RF", use_scaler=False).fit(Xn[tr][:, keep], y[tr]).predict(Xn[te][:, keep])
    rows.append((sg[te[0]], accuracy_score(y[te], p)))
r = pd.DataFrame(rows, columns=["signer", "acc"])
print(r.round(4)); print("LOSO tanpa slot2:", r["acc"].mean().round(4))

In [ ]:
import cv2, os
import mediapipe as mp
import matplotlib.pyplot as plt

mp_hands, mp_draw = mp.solutions.hands, mp.solutions.drawing_utils

def pick(cls, signer):
    folder = os.path.join(data_dir, cls)
    return os.path.join(folder, sorted(f for f in os.listdir(folder)
                        if f.startswith(signer + "_") and f.endswith(".mp4"))[0])

def show(path, ax, title):
    cap = cv2.VideoCapture(path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) // 2)
    ok, f = cap.read(); cap.release()
    with mp_hands.Hands(static_image_mode=True, max_num_hands=2) as h:
        res = h.process(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
    nh = len(res.multi_hand_landmarks) if res.multi_hand_landmarks else 0
    for lm in (res.multi_hand_landmarks or []):
        mp_draw.draw_landmarks(f, lm, mp_hands.HAND_CONNECTIONS)
    f = cv2.resize(f, None, fx=0.2, fy=0.2)
    ax.imshow(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)); ax.set_title(f"{title} | tangan={nh}"); ax.axis("off")

fig, axes = plt.subplots(2, 4, figsize=(14, 8))
for r, cls in enumerate(["Halo", "Sama-sama"]):
    for c, s in enumerate(["S5", "S1", "S2", "S7"]):
        show(pick(cls, s), axes[r, c], f"{s} - {cls}")
plt.tight_layout(); plt.show()

In [ ]:
def canon_hands(Xn):
    out = Xn.copy()
    h1, h2 = Xn[:, 99:162], Xn[:, 162:225]
    has1, has2 = np.abs(h1).sum(1) > 0, np.abs(h2).sum(1) > 0
    L, Rr = np.zeros_like(h1), np.zeros_like(h2)
    for i in range(len(Xn)):
        if has1[i] and has2[i]:
            a, b = (h1[i], h2[i]) if h1[i, 0] <= h2[i, 0] else (h2[i], h1[i])
            L[i], Rr[i] = a, b
        else:
            h = h1[i] if has1[i] else (h2[i] if has2[i] else None)
            if h is not None:
                (L if h[0] < 0 else Rr)[i] = h
    out[:, 99:162], out[:, 162:225] = L, Rr
    return out

Xc = canon_hands(Xn)
rows = []
for tr, te in LeaveOneGroupOut().split(Xc, y, groups=sg):
    p = make_classical_pipeline("RF", use_scaler=False).fit(Xc[tr], y[tr]).predict(Xc[te])
    rows.append((sg[te[0]], accuracy_score(y[te], p)))
r = pd.DataFrame(rows, columns=["signer", "acc"])
print(r.round(4)); print("LOSO (norm + urut tangan):", r["acc"].mean().round(4))

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import LeaveOneGroupOut

d = pd.read_csv("dataset_paper_5379_with_video.csv")
fc = [c for c in d.columns if c.startswith("f") and c[1:].isdigit()]
X, y, sg = d[fc].values, d["label"].values, d["signer"].values
mk = lambda: make_classical_pipeline("RF", use_scaler=False)

# 1. LOSO hanya di antara 6 pembelajar (S5 tidak ikut train maupun test)
L = sg != "S5"
rows = []
for tr, te in LeaveOneGroupOut().split(X[L], y[L], groups=sg[L]):
    p = mk().fit(X[L][tr], y[L][tr]).predict(X[L][te])
    rows.append((sg[L][te[0]], accuracy_score(y[L][te], p)))
r = pd.DataFrame(rows, columns=["signer", "acc"])
print(r.round(4)); print("LOSO 6 pembelajar:", r["acc"].mean().round(4), "+/-", r["acc"].std().round(4))

# 2. Arah sebaliknya: latih pada S5 saja, uji pada tiap pembelajar
U = sg == "S5"
m = mk().fit(X[U], y[U])
for s in sorted(set(sg[L])):
    mask = sg == s
    print(f"train S5 -> test {s}: {accuracy_score(y[mask], m.predict(X[mask])):.4f}")

In [ ]:
rows7 = []
for tr, te in LeaveOneGroupOut().split(X, y, groups=sg):
    p = mk().fit(X[tr], y[tr]).predict(X[te])
    acc = accuracy_score(y[te], p)
    f1 = f1_score(y[te], p, average="macro")
    rows7.append((sg[te[0]], acc, f1))
r7 = pd.DataFrame(rows7, columns=["signer", "acc", "f1_macro"])
print(r7.round(4))
print("LOSO 7 signer (semua):", r7["acc"].mean().round(4), "+/-", r7["acc"].std().round(4))

In [ ]:
import numpy as np, pandas as pd, time
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score

d = pd.read_csv("dataset_paper_5379_with_video.csv")
fc = [c for c in d.columns if c.startswith("f") and c[1:].isdigit()]
X, y = d[fc].values, d["label"].values
assert X.shape[1] == 285, f"Kolom fitur harus 285, ini {X.shape[1]}"

mk = lambda name, scaled: make_classical_pipeline(name, use_scaler=scaled)
RANDOM_STATE = 42  # samakan dengan RANDOM_STATE di notebook Anda kalau beda

# ============================================================
# C3 — Isolasi kontribusi fitur wajah (NMM proxy)
# ============================================================
print("=== C3: Isolasi fitur wajah ===")
pose_idx = slice(0, 99)
hand_idx = slice(99, 225)
face_idx = slice(225, 285)   # 20 facial keypoint x 3 = 60 dim

feature_sets = {
    "full (current)":          slice(0, 285),
    "no_face (skeleton_only)": np.r_[np.arange(0, 99), np.arange(99, 225)],
    "face_only":                face_idx,
}

c3_results = {}
for name, idx in feature_sets.items():
    scores = cross_val_score(mk("RF", False), X[:, idx], y, cv=5, scoring="accuracy")
    c3_results[name] = {"mean_acc": scores.mean(), "std": scores.std()}
    print(name, c3_results[name])

# fokus pasangan "Maaf" vs "Sama-sama"
pair_mask = np.isin(y, ["Maaf", "Sama-sama"])
Xp, yp = X[pair_mask], y[pair_mask]
print("\n-- Maaf vs Sama-sama saja --")
for name, idx in feature_sets.items():
    scores = cross_val_score(mk("RF", False), Xp[:, idx], yp, cv=5, scoring="accuracy")
    print(name, {"mean_acc": scores.mean(), "std": scores.std()})

pd.DataFrame(c3_results).T.to_csv("c3_facial_feature_ablation.csv")
print("Disimpan -> c3_facial_feature_ablation.csv\n")


# ============================================================
# C5 — Runtime & computational complexity
# ============================================================
print("=== C5: Runtime analysis ===")
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE)

def benchmark(pipe, X_train, y_train, X_test, n_repeats=5):
    t0 = time.perf_counter(); pipe.fit(X_train, y_train); train_time = time.perf_counter() - t0
    times = []
    for _ in range(n_repeats):
        t0 = time.perf_counter(); pipe.predict(X_test); times.append((time.perf_counter() - t0) / len(X_test))
    return {"train_time_s": train_time, "inference_ms_per_sample": np.mean(times) * 1000}

c5_results = {}
for name, scaled in [("RF", False), ("MLP", True), ("SVM", True), ("KNN", True)]:
    c5_results[name] = benchmark(mk(name, scaled), X_tr, y_tr, X_te)
    print(name, c5_results[name])
pd.DataFrame(c5_results).T.to_csv("c5_runtime_results.csv")
print("Disimpan -> c5_runtime_results.csv\n")


# ============================================================
# C6 — Bootstrap confidence interval
# ============================================================
print("=== C6: Bootstrap CI ===")
def bootstrap_ci(y_true, y_pred, n_boot=2000, ci=95, seed=42):
    rng = np.random.default_rng(seed)
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    n = len(y_true)
    accs = [accuracy_score(y_true[idx], y_pred[idx]) for idx in (rng.integers(0, n, n) for _ in range(n_boot))]
    return np.mean(accs), np.percentile(accs, 2.5), np.percentile(accs, 97.5)

c6_results = {}
for name, scaled in [("RF", False), ("MLP", True), ("SVM", True), ("KNN", True)]:
    pipe = mk(name, scaled).fit(X_tr, y_tr)
    y_pred = pipe.predict(X_te)
    mean_acc, lo, hi = bootstrap_ci(y_te, y_pred)
    c6_results[f"{name} (frame)"] = {"mean_acc": mean_acc, "ci_low": lo, "ci_high": hi}
    print(f"{name} (frame) 70:30: {mean_acc:.4f}  95% CI [{lo:.4f}, {hi:.4f}]")
pd.DataFrame(c6_results).T.to_csv("c6_bootstrap_ci_results.csv")
print("Disimpan -> c6_bootstrap_ci_results.csv")

print("\nSelesai. Kirim ketiga tabel (C3, C5, C6) yang tercetak di atas.")

In [ ]:
from sklearn.model_selection import StratifiedKFold

cv_fixed = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print("=== C3 (diperbaiki): Isolasi fitur wajah ===")
c3_fixed = {}
for name, idx in feature_sets.items():
    scores = cross_val_score(mk("RF", False), X[:, idx], y, cv=cv_fixed, scoring="accuracy")
    c3_fixed[name] = {"mean_acc": scores.mean(), "std": scores.std()}
    print(name, c3_fixed[name])

print("\n-- Maaf vs Sama-sama saja (diperbaiki) --")
for name, idx in feature_sets.items():
    scores = cross_val_score(mk("RF", False), Xp[:, idx], yp, cv=cv_fixed, scoring="accuracy")
    print(name, {"mean_acc": scores.mean(), "std": scores.std()})

pd.DataFrame(c3_fixed).T.to_csv("c3_facial_feature_ablation_fixed.csv")

In [ ]:
import os
print(os.path.exists("dataset_keypoints_with_video.csv"))

In [ ]:
import numpy as np, pandas as pd, os
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score

d = pd.read_csv("dataset_paper_5379_with_video.csv")
fc = [c for c in d.columns if c.startswith("f") and c[1:].isdigit()]
X, y = d[fc].values, d["label"].values
assert X.shape[1] == 285, f"Kolom fitur harus 285, ini {X.shape[1]}"

mk = lambda name, scaled: make_classical_pipeline(name, use_scaler=scaled)
cv_tune = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# ============================================================
# C4 — Hyperparameter tuning sistematis (GridSearchCV)
# ============================================================
print("=== C4: Hyperparameter tuning ===")

param_grids = {
    "RF":  {"clf__n_estimators": [50, 100, 200], "clf__max_depth": [None, 10, 20]},
    "MLP": {"clf__hidden_layer_sizes": [(128, 64), (256, 128)], "clf__alpha": [0.0001, 0.001]},
    "SVM": {"clf__C": [0.1, 1, 10], "clf__gamma": ["scale", 0.01]},
    "KNN": {"clf__n_neighbors": [3, 5, 7, 9]},
}

c4_results = {}
for name, scaled in [("RF", False), ("MLP", True), ("SVM", True), ("KNN", True)]:
    gs = GridSearchCV(mk(name, scaled), param_grids[name], cv=cv_tune, scoring="accuracy", n_jobs=-1)
    gs.fit(X, y)
    c4_results[name] = {"best_params": str(gs.best_params_), "best_cv_acc": gs.best_score_}
    print(name, c4_results[name])

pd.DataFrame(c4_results).T.to_csv("c4_hyperparameter_tuning_results.csv")
print("Disimpan -> c4_hyperparameter_tuning_results.csv\n")


# ============================================================
# C8 — Ablation study: threshold filtering & imputation
# ============================================================
print("=== C8: Ablation preprocessing ===")

RAW_CSV = "dataset_keypoints_with_video.csv"  # versi BELUM difilter, 6300 baris
assert os.path.exists(RAW_CSV), f"File {RAW_CSV} tidak ditemukan — ini harus versi 6300 baris sebelum threshold filtering."

def build_dataset_variant(raw_csv_path, threshold, impute="zero"):
    raw = pd.read_csv(raw_csv_path)
    feat_cols = [c for c in raw.columns if c.startswith("f") and c[1:].isdigit()]
    values = raw[feat_cols].values.astype(float)
    n_kpt = values.shape[1] // 3
    reshaped = values.reshape(values.shape[0], n_kpt, 3)
    missing_per_frame = np.sum((reshaped[:, :, 0] == 0) & (reshaped[:, :, 1] == 0), axis=1)

    keep_mask = missing_per_frame <= threshold
    df_variant = raw.loc[keep_mask].copy()

    if impute == "zero":
        df_variant[feat_cols] = df_variant[feat_cols].fillna(0.0)
    elif impute == "mean":
        for label in df_variant["label"].unique():
            mask = df_variant["label"] == label
            col_means = df_variant.loc[mask, feat_cols].replace(0, np.nan).mean(skipna=True)
            df_variant.loc[mask, feat_cols] = (
                df_variant.loc[mask, feat_cols].replace(0, np.nan).fillna(col_means)
            )
    return df_variant, feat_cols

ablation_configs = {
    "current (thr=30, zero-impute)":  dict(threshold=30, impute="zero"),
    "no_filtering (thr=inf)":         dict(threshold=10_000, impute="zero"),
    "stricter_filtering (thr=20)":    dict(threshold=20, impute="zero"),
    "looser_filtering (thr=40)":      dict(threshold=40, impute="zero"),
}

c8_results = {}
for name, cfg in ablation_configs.items():
    df_v, feat_cols = build_dataset_variant(RAW_CSV, **cfg)
    X_v, y_v = df_v[feat_cols].values, df_v["label"].values
    scores = cross_val_score(mk("RF", False), X_v, y_v, cv=5, scoring="accuracy")
    c8_results[name] = {"n_frames": len(df_v), "mean_acc": scores.mean(), "std": scores.std()}
    print(name, c8_results[name])

pd.DataFrame(c8_results).T.to_csv("c8_ablation_results.csv")
print("Disimpan -> c8_ablation_results.csv")

print("\nSelesai C4 + C8. Kirim kedua tabel yang tercetak di atas.")

In [ ]:
import numpy as np, pandas as pd, os
from sklearn.model_selection import StratifiedKFold, cross_val_score

RAW_CSV = "dataset_keypoints_with_video.csv"  # versi BELUM difilter, 6300 baris
assert os.path.exists(RAW_CSV), f"File {RAW_CSV} tidak ditemukan."

def build_dataset_variant(raw_csv_path, threshold, impute="zero"):
    raw = pd.read_csv(raw_csv_path)
    feat_cols = [c for c in raw.columns if c.startswith("f") and c[1:].isdigit()]
    values = raw[feat_cols].values.astype(float)
    n_kpt = values.shape[1] // 3
    reshaped = values.reshape(values.shape[0], n_kpt, 3)
    missing_per_frame = np.sum((reshaped[:, :, 0] == 0) & (reshaped[:, :, 1] == 0), axis=1)
    keep_mask = missing_per_frame <= threshold
    df_variant = raw.loc[keep_mask].copy()
    if impute == "zero":
        df_variant[feat_cols] = df_variant[feat_cols].fillna(0.0)
    return df_variant, feat_cols

ablation_configs = {
    "current (thr=30, zero-impute)":  dict(threshold=30, impute="zero"),
    "no_filtering (thr=inf)":         dict(threshold=10_000, impute="zero"),
    "stricter_filtering (thr=20)":    dict(threshold=20, impute="zero"),
    "looser_filtering (thr=40)":      dict(threshold=40, impute="zero"),
}

print("=== C8 (diperbaiki): Ablation preprocessing ===")
c8_results = {}
for name, cfg in ablation_configs.items():
    df_v, feat_cols = build_dataset_variant(RAW_CSV, **cfg)
    X_v, y_v = df_v[feat_cols].values, df_v["label"].values
    cv_fixed = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    scores = cross_val_score(mk("RF", False), X_v, y_v, cv=cv_fixed, scoring="accuracy")
    c8_results[name] = {"n_frames": len(df_v), "mean_acc": scores.mean(), "std": scores.std()}
    print(name, c8_results[name])

pd.DataFrame(c8_results).T.to_csv("c8_ablation_results_fixed.csv")
print("Disimpan -> c8_ablation_results_fixed.csv")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Times New Roman", "Liberation Serif", "DejaVu Serif"]

W, H = 7.0, 8.80
fig, ax = plt.subplots(figsize=(W, H))
ax.set_xlim(0, 100); ax.set_ylim(3.3, 127); ax.axis("off")

NAVY = "#1F3864"
STAGE_FILL = ["#DCE6F2", "#E4EFDC", "#FFF2CC", "#FBE4D5", "#E4DFEC"]
EDGE = "#404040"

def stage_box(y_top, h, num, title, fill):
    ax.add_patch(FancyBboxPatch((6, y_top - h), 88, h, boxstyle="round,pad=0.0,rounding_size=1.2",
                                fc=fill, ec=EDGE, lw=1.1, zorder=1))
    ax.add_patch(FancyBboxPatch((6, y_top - 5.2), 88, 5.2, boxstyle="round,pad=0.0,rounding_size=1.2",
                                fc=NAVY, ec=EDGE, lw=1.1, zorder=2))
    ax.text(9.5, y_top - 2.6, f"{num}", ha="center", va="center", fontsize=11, fontweight="bold",
            color=NAVY, zorder=4, bbox=dict(boxstyle="circle,pad=0.25", fc="white", ec="white"))
    ax.text(14, y_top - 2.6, title, ha="left", va="center", fontsize=10.5, fontweight="bold",
            color="white", zorder=4)

def inner(x, y, w, h, text, fs=8.6, ls=1.25):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.0,rounding_size=0.8",
                                fc="white", ec=EDGE, lw=0.8, zorder=3))
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=fs, zorder=4, linespacing=ls)

def inner2(x, y, w, h, title, body, fs=8.2, tfs=8.8, ls=1.3):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.0,rounding_size=0.8",
                                fc="white", ec=EDGE, lw=0.8, zorder=3))
    ax.text(x + w / 2, y + h - 1.3, title, ha="center", va="top", fontsize=tfs, fontweight="bold", zorder=4)
    ax.text(x + w / 2, y + h - 4.6, body, ha="center", va="top", fontsize=fs, zorder=4, linespacing=ls)

def arrow(x1, y1, x2, y2, lw=1.4, color=EDGE, ms=11):
    ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle="-|>", mutation_scale=ms,
                                 lw=lw, color=color, zorder=5, shrinkA=0, shrinkB=0))

# Stage 1
y1 = 125; h1 = 14
stage_box(y1, h1, 1, "Video Data Collection", STAGE_FILL[0])
inner(10, y1 - h1 + 1.6, 26, 6.4, "210 videos\n(35 per class)")
inner(37, y1 - h1 + 1.6, 26, 6.4, "6 phatic gesture classes\nHalo, Apa Kabar, Terima Kasih,\nSama-sama, Maaf, Tolong", fs=7.6)
inner(64, y1 - h1 + 1.6, 26, 6.4, "7 signers\n(1 Deaf teacher +\n6 hearing students)", fs=8.2)

# Stage 2
y2 = 107; h2 = 15
stage_box(y2, h2, 2, "Frame Extraction and Skeleton Keypoint Extraction", STAGE_FILL[1])
inner(10, y2 - h2 + 1.6, 36, 7.2, "Frame extraction\n30 evenly spaced frames per video\n(6,300 frames in total)", fs=8.3)
inner(54, y2 - h2 + 1.6, 36, 7.2, "MediaPipe Holistic\n33 pose + 42 hand + 20 face\n= 95 keypoints (x, y, z) = 285 dims", fs=8.1)
arrow(46.4, y2 - h2 + 5.2, 53.6, y2 - h2 + 5.2, lw=1.2, ms=9)

# Stage 3
y3 = 88; h3 = 17
stage_box(y3, h3, 3, "Data Quality Validation and Preprocessing", STAGE_FILL[2])
bw = 24.5; by = y3 - h3 + 2.0; bh = 8.6
inner(9.5, by, bw, bh, "Missing-keypoint analysis\n(heatmap and histogram)", fs=8.2)
inner(37.7, by, bw, bh, "Threshold filtering\n(> 30 missing keypoints\nremoved: 921 frames)", fs=8.0)
inner(65.9, by, bw, bh, "Zero imputation\n(fill value = 0.0)\nClean dataset:\n5,379 frames", fs=8.0)
arrow(9.5 + bw + 0.2, by + bh / 2, 37.7 - 0.2, by + bh / 2, lw=1.2, ms=9)
arrow(37.7 + bw + 0.2, by + bh / 2, 65.9 - 0.2, by + bh / 2, lw=1.2, ms=9)

# Stage 4
y4 = 67; h4 = 18.6
stage_box(y4, h4, 4, "Feature Engineering", STAGE_FILL[3])
bx_y = y4 - h4 + 4.4
inner(10, bx_y, 33, 7.0, "Frame-wise features\n(x, y, z of 95 keypoints)\n285 dimensions", fs=8.2)
inner(57, bx_y, 33, 7.0, "Delta + delta-delta appended\n(285 + 285 + 285)\n855 dimensions", fs=8.2)
arrow(43.3, bx_y + 3.5, 56.7, bx_y + 3.5, lw=1.3, ms=10)
ax.text(50, bx_y + 5.3, "temporal\ndifferences", ha="center", va="center", fontsize=7.2, style="italic", linespacing=1.1, zorder=5)
ax.text(50, y4 - h4 + 2.2, "Each classifier is evaluated on both representations", ha="center", va="center",
        fontsize=8.0, style="italic", zorder=5)

# Stage 5
y5 = 45.4; h5 = 40.5
stage_box(y5, h5, 5, "Model Training and Evaluation", STAGE_FILL[4])
top_row_y = y5 - 6.8 - 11.5
inner2(10, top_row_y, 38, 11.5, "Classical ML classifiers",
       "Random Forest (RF), Multilayer Perceptron (MLP),\nSupport Vector Machine (SVM),\nK-Nearest Neighbor (KNN)", fs=8.0)
inner2(52, top_row_y, 38, 11.5, "Deep learning baselines",
       "Video-level sequences (30 x 285)\n1D-CNN\nLSTM", fs=8.0)
ev_y = top_row_y - 3.6 - 17.2
ew = 25.2
inner2(10, ev_y, ew, 17.2, "Validation schemes",
       "Stratified 5-fold CV\nTrain-test split\n(80:20 and 70:30)\nLeave-one-signer-out\n(LOSO)", fs=7.9)
inner2(37.4, ev_y, ew, 17.2, "Metrics",
       "Accuracy\nMacro F1-score\nConfusion matrix\nRuntime\n(training / inference)", fs=7.9)
inner2(64.8, ev_y, ew, 17.2, "Statistical analysis",
       "Wilcoxon signed-rank\nCohen's d effect size\nBootstrap 95% CI\nTuning and ablation\nstudies", fs=7.9)
arrow(50, top_row_y - 0.1, 50, ev_y + 17.2 + 0.1, lw=1.4, ms=10)

# Arrows between stages
for (ya, yb) in [(y1 - h1, y2), (y2 - h2, y3), (y3 - h3, y4), (y4 - h4, y5)]:
    arrow(50, ya - 0.2, 50, yb + 0.2, lw=1.8, color=NAVY, ms=14)

plt.subplots_adjust(left=0.01, right=0.99, top=0.995, bottom=0.005)
fig.savefig("fig1_pipeline.png", dpi=300, facecolor="white")
fig.savefig("fig1_pipeline.pdf", facecolor="white")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Times New Roman", "Liberation Serif", "DejaVu Serif"]

# Nilai dan SD persis seperti Tabel 2 di manuskrip
models = ["RF", "MLP", "KNN", "SVM"]
frame = {"RF": (98.28, 0.58), "MLP": (98.21, 0.86), "KNN": (95.31, 0.74), "SVM": (94.90, 1.18)}
delta = {"RF": (97.79, 1.21), "MLP": (90.83, 1.08), "KNN": (84.28, 1.06), "SVM": (87.45, 1.19)}
dl = {"1D-CNN": (64.42, 10.22), "LSTM": (52.67, 5.54)}
colors = {"RF": "#4C78A8", "MLP": "#59A14F", "KNN": "#F28E2B", "SVM": "#E15759",
          "1D-CNN": "#8E6BB0", "LSTM": "#B8A0CF"}

fig, ax = plt.subplots(figsize=(7.2, 4.2))

bars = []
x = 0.0
for m in models:
    bars.append((x, *frame[m], colors[m], "", f"{m}\n(frame)")); x += 1
    bars.append((x, *delta[m], colors[m], "////", f"{m}\n(delta)")); x += 1.35
x += 0.2
for name in ["1D-CNN", "LSTM"]:
    bars.append((x, *dl[name], colors[name], "", name)); x += 1.2

for (xx, mean, sd, col, hatch, lab) in bars:
    ax.bar(xx, mean, width=0.82, color=col, edgecolor="#333333", linewidth=0.8, hatch=hatch, zorder=3)
    ax.errorbar(xx, mean, yerr=sd, fmt="none", ecolor="black", elinewidth=1.0, capsize=3, capthick=1.0, zorder=4)
    ax.text(xx, mean + sd + 1.2, f"{mean:.2f}", ha="center", va="bottom", fontsize=7.6, zorder=5)

ax.set_xticks([b[0] for b in bars])
ax.set_xticklabels([b[5] for b in bars], fontsize=8)
ax.set_ylabel("Mean accuracy (%)", fontsize=10)
ax.set_ylim(0, 108)
ax.set_yticks(range(0, 101, 20))
ax.tick_params(axis="y", labelsize=9)
ax.grid(axis="y", color="#DDDDDD", linewidth=0.7, zorder=0)
for sp in ("top", "right"):
    ax.spines[sp].set_visible(False)

legend = [Patch(fc="#BBBBBB", ec="#333333", label="Frame-wise features (285-dim)"),
          Patch(fc="#BBBBBB", ec="#333333", hatch="////", label="Delta features (855-dim)"),
          Patch(fc=colors["1D-CNN"], ec="#333333", label="Deep learning baselines")]
ax.legend(handles=legend, loc="lower center", bbox_to_anchor=(0.5, 1.01), ncol=3,
          fontsize=8, frameon=False, columnspacing=1.4, handlelength=1.6)

plt.tight_layout()
fig.savefig("fig5_model_comparison.png", dpi=300, facecolor="white")
fig.savefig("fig5_model_comparison.pdf", facecolor="white")
plt.show()